## 0. Preparación

In [ ]:
import json
import random
import re
import time
from datetime import date, datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Any

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from renewables_permitting.utils import (
    enum_value,
    normalize_text,
    safe_str,
    save_parquet,
    to_date_or_none,
    validate_required_columns,
)

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# BRONZE
BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"


# SILVER: candidatos BOE
BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"
BOE_CANDIDATES_DOCS_TEXT_PATH = SILVER_DIR / "boe_candidates_docs_text" / "boe_candidates_docs_text.parquet"

# SILVER: dimensiones
DIM_MUNICIPALITIES_PATH = SILVER_DIR / "dimensions" / "dim_municipalities.parquet"

# SILVER: extracción IA
SILVER_BOE_AI_DIR = SILVER_DIR / "boe_ai"

BOE_AI_EXTRACTIONS_PATH = SILVER_BOE_AI_DIR / "boe_ai_extractions.parquet"

PUBLICATION_EVENTS_PATH = SILVER_BOE_AI_DIR / "publication_events.parquet"
ADMINISTRATIVE_ACTIONS_PATH = SILVER_BOE_AI_DIR / "administrative_actions.parquet"
PROJECT_MENTIONS_PATH = SILVER_BOE_AI_DIR / "project_mentions.parquet"
PROJECT_TECHNICAL_ATTRIBUTES_PATH = SILVER_BOE_AI_DIR / "project_technical_attributes.parquet"
PROJECT_PARTICIPANTS_PATH = SILVER_BOE_AI_DIR / "project_participants.parquet"
PROJECT_LOCATIONS_PATH = SILVER_BOE_AI_DIR / "project_locations.parquet"
PROJECT_ALIASES_PATH = SILVER_BOE_AI_DIR / "project_aliases.parquet"
ASSOCIATED_INFRASTRUCTURE_PATH = SILVER_BOE_AI_DIR / "associated_infrastructure.parquet"

## 1. Contratos de salida

El contrato de salida óptimo estará desarrollado en tornon a cuatro ideas principales:

1. BOE publication: documento publicado.
2. Administrative event: acto o conjunto de actos administrativos publicados.
3. Project mention: proyecto energético afectado por ese acto.
4. Associated infrastructure summary: resumen textual de infraestructura auxiliar, no tabla exhaustiva de activos.

- Enum solo define valores permitidos.

- El idioma. Texto fuente del BOE → español. Categoría normalizada del sistema → inglés.
    - Python API names are in English.
    - Spanish BOE legal taxonomy values are stored as Spanish normalized slugs. El valor almacenado será jurídicamente próximo al BOE: "declaracion_impacto_ambiental"

### Clasificación documental

In [2]:
# Global energy relevance classification for a BOE publication.
# Enum values intentionally use Spanish normalized slugs because the source
# documents are BOE publications written in Spanish.

class EnergyRelevance(str, Enum):
    RELEVANT = "relevante"
    NOT_RELEVANT = "no_relevante"
    UNCERTAIN = "dudoso"


# relevante:
#   BOE publication concerning renewable electricity generation, energy storage,
#   evacuation infrastructure, electrical substations, power lines, or related
#   environmental/administrative authorizations for a specific energy project.

# no_relevante:
#   Generic energy-related, regulatory, statistical, tariff, budgetary, or
#   non-project-specific BOE publication.

# dudoso:
#   BOE publication containing energy-related terminology, but without enough
#   information to determine whether it concerns a specific project under
#   administrative processing.

### Tecnologías e instalaciones energéticas

In [3]:
# Tecnologías de generación y almacenamiento.
# technology_type = qué tipo de instalación o tecnología es
# EnergyInstallationType no debe ser una copia literal de las palabras encontradas en el BOE. Debe ser una taxonomía controlada para clasificar semánticamente lo que aparece en el BOE.

class EnergyInstallationType(str, Enum):
    PHOTOVOLTAIC = "fotovoltaica"
        # PHOTOVOLTAIC  -> identificador técnico del código
        # "fotovoltaica" -> valor normalizado alineado con el BOE
    WIND = "eolica"
    CONCENTRATED_SOLAR_POWER = "termosolar"
    HYDROPOWER = "hidroelectrica"
    GEOTHERMAL = "geotermica"
    BIOMASS = "biomasa"
    BIOGAS = "biogas"
    GREEN_HYDROGEN = "hidrogeno_verde"

    # Use these only when they are the primary subject of the BOE publication,
    # or when they are necessary to identify the project being processed.
    ENERGY_STORAGE = "almacenamiento"
    EVACUATION_INFRASTRUCTURE = "infraestructura_evacuacion"
    ELECTRICAL_SUBSTATION = "subestacion_electrica"
    POWER_LINE = "linea_electrica"

    OTHER = "otra"
    UNKNOWN = "desconocido"

In [4]:
# Características técnicas de la tecnología.
class EnergyInstallationTechnicalAttributes(BaseModel):
    installation_type: EnergyInstallationType

    installed_power_mw: float | None = None
    peak_power_mwp: float | None = None

    storage_power_mw: float | None = None
    storage_capacity_mwh: float | None = None

    description: str | None = None
    power_normalization_note: str | None = None
    evidence: str | None = None

### Localización administrativa (INE)

Usar Pydantic AI solo para extraer candidatos textuales y contexto; la validación final debe hacerla una función determinista.

In [5]:
# Pydantic AI should only extract textual municipality candidates and context.
# Final validation and INE code resolution must be performed by a deterministic
# post-processing function.

class ExtractedMunicipalityMention(BaseModel):
    municipality_name: str
    province_hint: str | None = None
    autonomous_community_hint: str | None = None
    evidence: str | None = None

### Procedimiento administrativo

- A la IA no le pediremos el "estado actual del proyecto" ya que este se derivará después, ordenando los eventos por fechas y aplicando reglas deterministas.

- Sobre impacto ambiental:
    - En la Ley 21/2013, la declaración de impacto ambiental (DIA) es el informe con el que concluye la evaluación de impacto ambiental ordinaria, mientras que el informe de impacto ambiental concluye la evaluación simplificada.

    - evaluacion_impacto_ambiental
        → procedimiento ambiental genérico.

    - declaracion_impacto_ambiental
        → resolución ambiental final de la evaluación ordinaria; normalmente aparece como DIA.

    - informe_determinacion_afeccion_ambiental
        → informe específico, normalmente IDAA, frecuente en tramitaciones de renovables.

In [6]:
# Tipo de trámite administrativo.
# Administrative procedure
# The AI must not infer the current consolidated status of the project.
# The current status will be derived later by ordering publication events by date
# and applying deterministic rules.

class ProcedureStage(str, Enum):
    # Inicio y antecedentes del procedimiento
    APPLICATION_SUBMISSION = "solicitud_tramitacion"
    ENVIRONMENTAL_APPLICATION_SUBMISSION = "solicitud_tramitacion_ambiental"
    DOCUMENTATION_CORRECTION = "subsanacion_documentacion"
    REQUIREMENTS_VERIFICATION = "verificacion_requisitos_tramitacion"

    # Información pública
    PUBLIC_INFORMATION = "informacion_publica"

    # Evaluación ambiental
    ENVIRONMENTAL_IMPACT_ASSESSMENT = "evaluacion_impacto_ambiental"
    # Producto de la evaluación ambiental
    ENVIRONMENTAL_IMPACT_STATEMENT = "declaracion_impacto_ambiental"
    ENVIRONMENTAL_IMPACT_REPORT = "informe_impacto_ambiental"
    ENVIRONMENTAL_IMPACT_DETERMINATION_REPORT = (
        "informe_determinacion_afeccion_ambiental"
    )

    # Autorizaciones energéticas
    PRIOR_ADMINISTRATIVE_AUTHORIZATION = "autorizacion_administrativa_previa"
    CONSTRUCTION_ADMINISTRATIVE_AUTHORIZATION = (
        "autorizacion_administrativa_construccion"
    )
    OPERATING_AUTHORIZATION = "autorizacion_explotacion"

    # Utilidad pública y expropiación
    PUBLIC_UTILITY_DECLARATION = "declaracion_utilidad_publica"
    FORCED_EXPROPRIATION = "expropiacion_forzosa"
    AFFECTED_ASSETS_AND_RIGHTS_LIST = "relacion_bienes_derechos_afectados"
    PRIOR_OCCUPATION_RECORDS = "levantamiento_actas_previas_ocupacion"
    OCCUPATION_RECORDS = "actas_ocupacion"

    # Modificaciones y terminación anormal
    MODIFICATION = "modificacion"
    DEADLINE_EXTENSION = "prorroga"
    CASE_FILE_CLOSURE = "archivo_expediente"
    WITHDRAWAL = "desistimiento"
    INADMISSIBILITY = "inadmision"
    OWNERSHIP_CHANGE = "cambio_titularidad"

    # Fallback
    OTHER = "otro"
    UNKNOWN = "desconocido" 

In [7]:
# Resultado o decisión administrativa.
class ProcedureDecision(str, Enum):
    # Inicio y tramitación del expediente
    REQUESTED = "solicitado"
    CORRECTED = "subsanado"
    REQUIREMENTS_VERIFIED = "requisitos_verificados"
    MODIFIED = "modificado"
    EXTENDED = "prorrogado"
    ISSUED = "formulado"

    # Información pública y participación
    SUBMITTED_TO_PUBLIC_INFORMATION = "sometido_informacion_publica"
    ANNOUNCED = "convocado"

    # Evaluación ambiental: valoración general
    FAVORABLE = "favorable"
    UNFAVORABLE = "desfavorable"
    ENVIRONMENTALLY_VIABLE = "ambientalmente_viable"
    ENVIRONMENTALLY_NOT_VIABLE = "ambientalmente_no_viable"

    # Evaluación ambiental: necesidad de evaluación ambiental adicional
    FURTHER_ENVIRONMENTAL_ASSESSMENT_REQUIRED = (
        "requiere_evaluacion_ambiental_adicional"
    )
    FURTHER_ENVIRONMENTAL_ASSESSMENT_NOT_REQUIRED = (
        "no_requiere_evaluacion_ambiental_adicional"
    )

    # Utilidad pública
    PUBLIC_UTILITY_DECLARED = "declarado_utilidad_publica"

    # Resolución administrativa
    APPROVED = "aprobado"
    AUTHORIZED = "autorizado"
    DENIED = "denegado"

    # Terminación anormal del procedimiento
    CLOSED = "archivado"
    WITHDRAWN = "desistido"
    INADMISSIBLE = "inadmitido"

    # Fallback
    UNKNOWN = "desconocido"

In [8]:
# Acto administrativo publicado en el BOE.
class AdministrativeAction(BaseModel):
    procedure_stage: ProcedureStage = ProcedureStage.UNKNOWN
    procedure_decision: ProcedureDecision = ProcedureDecision.UNKNOWN
    evidence: str | None = None

### Participantes

In [9]:
# Rol de una entidad participante.
class ParticipantRole(str, Enum):
    PROMOTER = "promotor"
    CO_PROMOTER = "copromotor"
    HOLDER = "titular"
    OPERATOR = "operador"

    GRID_OPERATOR = "gestor_red"
    GRID_OWNER = "propietario_red"

    COMPETENT_AUTHORITY = "organo_sustantivo"
    ENVIRONMENTAL_AUTHORITY = "organo_ambiental"
    ADMINISTRATION = "administracion"

    OTHER = "otro"
    UNKNOWN = "desconocido"

In [10]:
# Empresa, administración o entidad participante.
class ProjectParticipant(BaseModel):
    name: str
    role: ParticipantRole = ParticipantRole.UNKNOWN
    evidence: str | None = None

### Evolución del proyecto y estado mencionado en el documento

In [11]:
# Tipo de evolución material del proyecto.
# event_type = qué tipo de evolución/acto material representa el evento
class LifecycleEventType(str, Enum):
    NEW_PROJECT = "proyecto_nuevo"
    HYBRIDIZATION = "hibridacion"
    MODIFICATION = "modificacion"
    REPOWERING = "repotenciacion"
    STORAGE_ADDITION = "incorporacion_almacenamiento"
    STANDALONE_INFRASTRUCTURE_PROJECT = "proyecto_infraestructura_autonoma"
    OWNERSHIP_CHANGE = "cambio_titularidad"
    TERMINATION = "terminacion"
    OTHER = "otro"
    UNKNOWN = "desconocido"

MODIFICATION
- Incluye modificaciones técnicas, cambios de potencia, cambios de configuración
  o ampliaciones si no quieres distinguirlas como categoría separada.

STORAGE_ADDITION
- Úsalo cuando el evento principal sea añadir almacenamiento a una instalación
  existente.

STANDALONE_INFRASTRUCTURE_PROJECT
- Úsalo solo cuando el objeto principal del BOE sea una línea, subestación,
  infraestructura de evacuación o infraestructura eléctrica autónoma.
- No lo uses cuando esa infraestructura sea auxiliar de un parque o planta;
  en ese caso debe ir en AssociatedInfrastructureSummary.

TERMINATION
- Agrupa archivo, desistimiento, inadmisión o denegación cuando afectan al
  proyecto.

In [12]:
# role_in_event = qué papel tiene en este BOE
# PRIMARY_SUBJECT
# - Proyecto o instalación directamente objeto del expediente BOE.
# - Puede ser un parque, planta, BESS, línea, subestación o infraestructura autónoma.

# EXISTING_REFERENCE
# - Instalación existente mencionada para entender el evento.
# - Muy útil en hibridaciones, almacenamiento asociado o modificaciones.

# ASSOCIATED_REFERENCE
# - Proyecto relacionado, compartido o citado como contexto, pero no objeto principal del acto.

# UNKNOWN
# - No se puede determinar el papel con suficiente seguridad.

class ProjectRoleInEvent(str, Enum):
    PRIMARY_SUBJECT = "objeto_principal"
    EXISTING_REFERENCE = "referencia_existente"
    ASSOCIATED_REFERENCE = "referencia_asociada"
    UNKNOWN = "desconocido"

In [13]:
# Status explicitly mentioned or strongly implied in this BOE document.
#
# This is not the final consolidated project status. It is only the local status
# inferred from this specific publication.

class ProjectStatusInDocument(str, Enum):
    PLANNED = "proyectado"
    UNDER_PERMITTING = "en_tramitacion"
    AUTHORIZED = "autorizado"
    UNDER_CONSTRUCTION = "en_construccion"
    IN_OPERATION = "en_explotacion"
    EXISTING = "existente"
    DENIED = "denegado"
    CLOSED = "archivado"
    UNKNOWN = "desconocido"

In [14]:
class EnergyProjectMention(BaseModel):
    local_project_id: str
    name: str | None = None
    aliases: list[str] = Field(default_factory=list)

    role_in_event: ProjectRoleInEvent = ProjectRoleInEvent.UNKNOWN
    status_in_document: ProjectStatusInDocument = ProjectStatusInDocument.UNKNOWN

    technical_attributes: list[EnergyInstallationTechnicalAttributes] = Field(
        default_factory=list
    )

    participants: list[ProjectParticipant] = Field(default_factory=list)
    locations: list[ExtractedMunicipalityMention] = Field(default_factory=list)

    case_file_reference: str | None = None

    evidence: str | None = None

### Infraestructura asociada

Si la infraestructura es auxiliar del proyecto principal → AssociatedInfrastructureSummary.

Si la infraestructura es el objeto principal del BOE → EnergyProjectMention con installation_type = linea_electrica, subestacion_electrica o infraestructura_evacuacion.

In [15]:
# La infraestructura asociada puede que no sea exhaustiva.

# Summary of auxiliary infrastructure associated with the project.
#
# This block is intentionally non-exhaustive. It should preserve relevant
# information about evacuation, substations, grid connection or shared
# infrastructure without forcing the AI to extract each line, substation or
# electrical component as a separate project mention.

class AssociatedInfrastructureSummary(BaseModel):
    has_evacuation_infrastructure: bool | None = None
    has_electrical_substation: bool | None = None
    has_grid_connection: bool | None = None
    has_shared_infrastructure: bool | None = None

    description: str | None = None
    evidence: str | None = None

has_evacuation_infrastructure
- True si se menciona infraestructura de evacuación: líneas de evacuación, red colectora, infraestructura común de evacuación, etc.

has_electrical_substation
- True si se mencionan SET, SE, subestaciones colectoras, subestaciones elevadoras o similares.

has_grid_connection
- True si se menciona punto de conexión, conexión a red de transporte/distribución, REE, red de transporte, red de distribución, posición de conexión, etc.

has_shared_infrastructure
- True si la infraestructura parece compartida, colectora o común a varios proyectos.

### Evento administrativo publicado

- PublicationEvent → representa el hecho administrativo principal publicado en el BOE. Puede afectar a uno o varios ProjectMention.
- ProjectLifecycle → lo que reconstruyes después al agrupar varios PublicationEvent del mismo proyecto.

In [16]:
# PublicationEvent representa lo que publica un BOE concreto.

# A PublicationEvent represents the main administrative event published in a BOE
# document. It may affect one or more EnergyProjectMention objects.
#
# This is not the full lifecycle of a project. The full project lifecycle will be
# reconstructed later by grouping multiple BOE publication events that refer to
# the same project.

class PublicationEvent(BaseModel):
    event_type: LifecycleEventType = LifecycleEventType.UNKNOWN

    administrative_actions: list[AdministrativeAction] = Field(default_factory=list)

    project_mentions: list[EnergyProjectMention] = Field(default_factory=list)

    associated_infrastructure: AssociatedInfrastructureSummary | None = None

    event_summary: str | None = None
    evidence: str | None = None

event_type
- Tipo de evolución material o administrativa del proyecto.
- Ejemplos: proyecto_nuevo, hibridacion, modificacion, incorporacion_almacenamiento, cambio_titularidad, terminacion.

administrative_actions
- Actos administrativos concretos publicados.
- Ejemplos: información pública, DIA, AAP, AAC, DUP, archivo, desistimiento.

project_mentions
- Proyecto o proyectos afectados por el evento.
- Ejemplos: parque eólico, planta fotovoltaica, BESS, línea eléctrica autónoma, subestación autónoma.

associated_infrastructure
- Infraestructura auxiliar mencionada, sin descomponerla exhaustivamente.

event_summary
- Resumen breve del evento. Debe estar en español, preferiblemente con terminología próxima al BOE.

evidence
- Fragmento textual del BOE que justifica el evento.

### Documento raíz (agrega todo)

In [17]:
# Complete structured extraction for one BOE publication.
#
# This object represents the information extracted from a single BOE document.
# It does not represent a consolidated energy project. Project consolidation,
# deduplication, timeline reconstruction and current status inference must be
# performed later through deterministic post-processing.

class BOEProjectExtraction(BaseModel):
    boe_id: str
    publication_date: date | None = None

    energy_relevance: EnergyRelevance
    is_project_specific: bool
    relevance_reason: str | None = None

    publication_events: list[PublicationEvent] = Field(default_factory=list)

    extraction_notes: str | None = None

#### La IA debe extraer información suficiente para responder

* ¿Qué **publicación BOE** se está procesando?
* ¿Qué **evento administrativo publicado** contiene?
* ¿Qué **acto o trámite administrativo** se publica?
* ¿Qué **decisión administrativa o ambiental** se adopta o comunica?
* ¿Qué **proyecto energético** aparece?
* ¿Qué papel tiene ese proyecto en el documento: **objeto principal**, **referencia existente** o **referencia asociada**?
* ¿Dónde se localiza textualmente el proyecto?
* ¿Quién lo promueve, ostenta la titularidad, opera o administra?
* Qué **tipo de instalación o tecnología** aparece: eólica, fotovoltaica, almacenamiento, línea eléctrica, subestación, infraestructura de evacuación, etc.
* Qué **potencia o capacidad** se menciona.
* Qué **infraestructura auxiliar relevante** se menciona, sin descomponerla exhaustivamente.

#### La IA no debe intentar responder todavía

* ¿Cuál es el **estado actual consolidado** del proyecto?
* Qué publicaciones BOE pertenecen al **mismo proyecto consolidado**.
* Qué proyectos deben agruparse bajo un mismo `project_group_id`.
* Qué subestaciones, líneas y posiciones forman una **topología eléctrica completa**.
* Cuál es el **código INE definitivo** de cada municipio.
* Cuál es el **CIF definitivo** del promotor o titular.
* Cuál es la situación administrativa final si requiere combinar varias publicaciones BOE.
* Qué infraestructura auxiliar debe modelarse como activo independiente, salvo que sea el **objeto principal** del expediente.


#### Razón de la estructura del contrato de extracción

La estructura del contrato separa tres niveles distintos: el **documento BOE**, los **eventos publicados** dentro de ese documento y las **entidades o datos asociados** a cada evento.

```text
BOEProjectExtraction
└── publication_events
    ├── administrative_actions
    ├── project_mentions
    │   ├── technical_attributes
    │   ├── participants
    │   ├── locations
    │   └── aliases
    └── associated_infrastructure
```

##### 1. Documento BOE: `BOEProjectExtraction`

`BOEProjectExtraction` representa la extracción completa de una **publicación BOE concreta**. No representa un proyecto energético consolidado.

Es el contenedor principal de la extracción e incluye metadatos documentales como:

* `boe_id`: identificador del BOE.
* `publication_date`: fecha de publicación.
* `energy_relevance`: relevancia energética del documento.
* `is_project_specific`: indica si el documento se refiere a uno o varios proyectos concretos.
* `publication_events`: lista de eventos publicados en ese documento.

Por ejemplo, para un documento como:

```text
BOE-B-2024-26379
```

la extracción completa pertenece a ese único documento, aunque dentro puedan mencionarse varios proyectos, una infraestructura de evacuación o distintas actuaciones administrativas.

---

##### 2. Eventos publicados: `publication_events`

`publication_events` representa los **hechos administrativos principales publicados** en el documento BOE.

Un `publication_event` no es todavía el ciclo de vida completo del proyecto. Es solo lo que ese documento concreto publica. Un BOE puede contener uno o varios eventos, aunque en la mayoría de los casos habrá uno.

Por ejemplo, un BOE puede publicar:

```text
Información pública de la solicitud de autorización administrativa previa y autorización administrativa de construcción.
```

Eso sería un único `publication_event`, porque el BOE publica una unidad administrativa-documental: la información pública de una solicitud. Dentro de ese evento habría dos `administrative_actions`, una relativa a la autorización administrativa previa y otra relativa a la autorización administrativa de construcción.

La consolidación de varios eventos BOE en una cronología completa debe hacerse después, en la fase **gold** o de agrupación.

---

##### 3. Actos administrativos: `administrative_actions`

Dentro de cada `publication_event`, `administrative_actions` recoge los **actos administrativos formales** incluidos en ese evento.

Esto permite separar el evento publicado de sus trámites o decisiones internas.

Ejemplos de actos administrativos:

```text
Información pública
Declaración de impacto ambiental
Autorización administrativa previa
Autorización administrativa de construcción
Declaración de utilidad pública
Archivo
Desistimiento
Inadmisión
Denegación
```

Por ejemplo, un mismo evento puede contener dos acciones:

```text
Autorización administrativa previa: otorgada
Autorización administrativa de construcción: otorgada
```

En ese caso, el evento publicado es uno, pero contiene dos `administrative_actions`.

---

##### 4. Menciones de proyecto: `project_mentions`

`project_mentions` recoge las menciones a proyectos o instalaciones energéticas sustantivas dentro del evento.

Se usa el término **mención** porque todavía no se afirma que sea un proyecto consolidado único dentro de toda la base de datos. Solo se registra que en ese BOE aparece una referencia explícita a una instalación o proyecto.

Por ejemplo:

```text
BESS Hibridación FV Andévalo
FV Andévalo
FV La Puebla 1
```

pueden aparecer como menciones en documentos relacionados, pero no deben agruparse automáticamente solo porque compartan infraestructura, municipio o contexto administrativo.

Esa decisión debe hacerse después mediante reglas deterministas de agrupación.

`EnergyProjectMention` no debe usarse para extraer cada línea, subestación o componente auxiliar, salvo que esa infraestructura sea el objeto principal del expediente.

---

##### 5. Atributos técnicos: `technical_attributes`

Dentro de cada `project_mention`, `technical_attributes` recoge las características técnicas de la instalación mencionada: tipo de instalación, potencia instalada, potencia pico, potencia o capacidad de almacenamiento, descripción técnica y notas de normalización.

Ejemplo de almacenamiento:

```text
installation_type = "almacenamiento"
storage_power_mw = 26.36
```

Ejemplo de instalación fotovoltaica:

```text
installation_type = "fotovoltaica"
installed_power_mw = 42.56
```

Esta separación permite que una misma mención tenga varios bloques técnicos, por ejemplo una planta fotovoltaica y un sistema de almacenamiento asociado.

---

##### 6. Participantes: `participants`

`participants` recoge entidades asociadas a una mención concreta de proyecto: promotor, titular, órgano administrativo u otros participantes relevantes.

Ejemplo:

```text
participant_name = "Iberdrola Renovables Andalucía, S.A.U."
participant_role = "promotor"
```

La extracción registra la entidad tal como aparece en el BOE. La resolución o normalización empresarial puede hacerse después en otra fase.

Los participantes deben asignarse únicamente a la `project_mention` a la que el texto los vincule explícitamente. No deben propagarse automáticamente a menciones secundarias, antecedentes, proyectos existentes o instalaciones asociadas si el documento no los vincula de forma clara.

Por ejemplo, si el BOE indica que un sistema de almacenamiento está promovido por una sociedad determinada y está asociado a una planta fotovoltaica existente, el promotor debe asignarse al sistema de almacenamiento, no necesariamente a la planta fotovoltaica existente.

---

##### 7. Localizaciones: `locations`

`locations` recoge las localizaciones textuales mencionadas en el BOE y asociadas a una mención concreta de proyecto.

En esta fase no se guarda todavía el código **INE** (Instituto Nacional de Estadística) definitivo, porque la resolución municipal debe hacerse después mediante un proceso determinista.

Ejemplo:

```text
municipality_raw = "Puebla de Guzmán"
province_hint_raw = "Huelva"
```

Después, en otra fase, esa mención se resolverá contra una dimensión oficial de municipios.

Igual que ocurre con los participantes, las localizaciones no deben propagarse automáticamente a todas las menciones del evento. Solo deben asignarse a la mención de proyecto a la que el texto las vincule explícitamente.

---

##### 8. Alias o denominaciones alternativas: `aliases`

`aliases` recoge denominaciones alternativas del mismo proyecto dentro del documento.

Por ejemplo:

```text
"BESS Hibridación FV Andévalo"
"Hibridación FV Andévalo"
```

Esto es útil porque los proyectos pueden aparecer con nombres ligeramente distintos en varios documentos del BOE.

---

##### 9. Infraestructura asociada: `associated_infrastructure`

`associated_infrastructure` recoge una síntesis no exhaustiva de infraestructura auxiliar mencionada en el evento: evacuación, subestaciones, conexión a red o infraestructura compartida.

Ejemplo:

```text
has_evacuation_infrastructure = True
has_shared_infrastructure = True
description = "infraestructura común de evacuación asociada a varias instalaciones"
```

Si la infraestructura es **auxiliar** del proyecto principal, debe resumirse en `AssociatedInfrastructureSummary`.

Si la infraestructura es el **objeto principal** del BOE, debe extraerse como `EnergyProjectMention`, con el `installation_type` correspondiente:

```text
linea_electrica
subestacion_electrica
infraestructura_evacuacion
```

Esta parte no intenta crear proyectos separados para cada línea, subestación o posición eléctrica auxiliar. Solo indica que el evento menciona infraestructura asociada relevante.

Si más adelante se quisiera modelar esa infraestructura como entidad propia, habría que definir un contrato más detallado.

---

##### 10. Estado del proyecto: `status_in_document`

`status_in_document` recoge únicamente el estado mencionado o inferible en esa publicación concreta.

No representa el estado actual consolidado del proyecto.

El estado actual debe derivarse después, agrupando todas las publicaciones BOE de un mismo proyecto, ordenándolas cronológicamente y aplicando reglas deterministas.

Por ejemplo, una publicación puede indicar que un proyecto está:

```text
en_tramitacion
```

y otra publicación posterior puede indicar que está:

```text
autorizado
```

La extracción IA no debe resolver por sí sola el estado consolidado. Solo debe registrar el estado correspondiente al documento analizado.

---

##### 11. Regla sobre idioma

Los nombres de clases y campos están en inglés para mantener consistencia técnica en el código.

Los valores normalizados de los `Enum` se mantienen en español canónico sin tildes cuando representan categorías propias del BOE o del procedimiento administrativo español.

Por ejemplo:

```text
PLANNED = "proyectado"
UNDER_PERMITTING = "en_tramitacion"
AUTHORIZED = "autorizado"
```

Los campos textuales extraídos del BOE deben conservarse en español y respetar la evidencia documental:

* `name`
* `aliases`
* `description`
* `power_normalization_note`
* `event_summary`
* `relevance_reason`
* `extraction_notes`
* `evidence`

---

##### Razón metodológica

La razón principal de esta estructura es evitar mezclar niveles conceptuales:

* Un **BOE** no es un proyecto.
* Un **evento publicado** no es necesariamente todo el ciclo de vida del proyecto.
* Una **mención de proyecto** no es todavía un proyecto consolidado.
* Una **infraestructura asociada** no debe provocar por sí sola que dos proyectos se agrupen.
* El **estado mencionado en un documento** no es necesariamente el estado actual del proyecto.

Así, la extracción con IA se limita a registrar lo que aparece explícitamente en cada documento. La agrupación de menciones en proyectos reales, la resolución de municipios, la normalización de entidades y la construcción de cronologías administrativas quedan para fases posteriores, más controladas y deterministas.


```text
BOEProjectExtraction
│
├── boe_id
├── publication_date
├── energy_relevance
├── is_project_specific
├── relevance_reason
├── extraction_notes
│
└── publication_events
    │
    └── PublicationEvent
        │
        ├── event_type
        ├── event_summary
        ├── evidence
        │
        ├── administrative_actions
        │   │
        │   └── AdministrativeAction
        │       ├── procedure_stage
        │       ├── procedure_decision
        │       └── evidence
        │
        ├── project_mentions
        │   │
        │   └── EnergyProjectMention
        │       ├── local_project_id
        │       ├── name
        │       ├── aliases
        │       ├── role_in_event
        │       ├── status_in_document
        │       │
        │       ├── technical_attributes
        │       │   │
        │       │   └── EnergyInstallationTechnicalAttributes
        │       │       ├── installation_type
        │       │       ├── installed_power_mw
        │       │       ├── peak_power_mwp
        │       │       ├── storage_power_mw
        │       │       ├── storage_capacity_mwh
        │       │       ├── description
        │       │       ├── power_normalization_note
        │       │       └── evidence
        │       │
        │       ├── participants
        │       │   │
        │       │   └── ProjectParticipant
        │       │       ├── name
        │       │       ├── role
        │       │       └── evidence
        │       │
        │       ├── locations
        │       │   │
        │       │   └── ExtractedMunicipalityMention
        │       │       ├── municipality_name
        │       │       ├── province_hint
        │       │       ├── autonomous_community_hint
        │       │       └── evidence
        │       │
        │       ├── case_file_reference
        │       └── evidence
        │
        └── associated_infrastructure
            │
            └── AssociatedInfrastructureSummary
                ├── has_evacuation_infrastructure
                ├── has_electrical_substation
                ├── has_grid_connection
                ├── has_shared_infrastructure
                ├── description
                └── evidence
```

## 2. Instrucciones

El objetivo final no es saber cuántas subestaciones, líneas o posiciones eléctricas hay, sino reconstruir para cada proyecto energético, un cronología como:
````
Proyecto X
- 2021-07-07: información pública AAP/AAC/DUP
- 2023-01-31: DIA favorable
- 2023-04-28: autorización administrativa previa
- 2024-...: autorización administrativa de construcción
- estado actual inferido: autorizado / en tramitación / archivado / denegado / etc.`
````

La solución óptima sería diseñar el contrato en torno a cuatro ideas:
- BOE publication: documento publicado.
- Administrative event: acto o conjunto de actos administrativos publicados.
- Project mention: proyecto energético afectado por ese acto.
- Associated infrastructure summary: resumen textual de infraestructura auxiliar, no tabla exhaustiva de activos.

In [18]:
INSTRUCTIONS = """
Eres un extractor canónico de información estructurada de documentos del BOE sobre proyectos energéticos.

Devuelve exclusivamente JSON válido conforme al esquema BOEProjectExtraction.
No incluyas explicaciones fuera del JSON.

Objetivo:
Extraer, desde una única publicación BOE, información suficiente para reconstruir posteriormente la evolución administrativa de proyectos energéticos mediante reglas deterministas.

Prioridad de reglas:
1. Cumple estrictamente el esquema BOEProjectExtraction.
2. Extrae solo información respaldada por el texto del BOE.
3. Usa los valores enum definidos por el contrato.
4. Si existe ambigüedad, conserva la evidencia y no fuerces una interpretación.

Unidad de extracción:
- BOEProjectExtraction representa una publicación BOE concreta.
- publication_events representa el hecho administrativo principal publicado en esa publicación.
- administrative_actions recoge los actos administrativos publicados.
- project_mentions recoge proyectos o instalaciones energéticas sustantivas afectadas.
- associated_infrastructure resume infraestructura auxiliar relevante, sin descomponerla exhaustivamente.
- No reconstruyas el ciclo de vida completo del proyecto.
- No agrupes publicaciones BOE.
- No generes identificadores globales.
- Usa local_project_id internos: project_1, project_2, project_3.

Reglas generales:
- Extrae únicamente información explícita del título o texto del BOE.
- No inventes, completes, corrijas ni resuelvas información con conocimiento externo.
- Si un dato opcional no aparece, usa null.
- Si una lista no tiene elementos, usa [].
- Usa "desconocido" solo en campos enum cuando la clasificación no pueda determinarse.
- No escribas "desconocido", "no consta" o "no aplica" en campos textuales salvo que aparezcan literalmente en el BOE.
- Toda información relevante debe tener evidence textual específica.
- Distingue el hecho publicado de los antecedentes históricos.
- Los antecedentes no deben crear publication_events ni administrative_actions salvo que formen parte del acto publicado actual.
- Los campos textuales deben conservarse en español: name, aliases, description, power_normalization_note, event_summary, relevance_reason, extraction_notes y evidence.

Relevancia:
- energy_relevance = "relevante" si el documento trata de un proyecto o infraestructura energética concreta.
- energy_relevance = "no_relevante" si es normativo, estadístico, tarifario, presupuestario, genérico o no vinculado a un proyecto concreto.
- energy_relevance = "dudoso" si contiene vocabulario energético pero no permite identificar un proyecto o infraestructura concreta.
- is_project_specific debe ser true solo si se identifica un proyecto, instalación o infraestructura energética concreta.
- relevance_reason debe justificar brevemente la clasificación.

PublicationEvent:
- Cada publicación BOE debe generar normalmente un único PublicationEvent.
- Crea varios PublicationEvent solo si hay hechos administrativos principales independientes sobre proyectos o instalaciones sustantivas distintas.
- No crees un PublicationEvent separado para cada trámite si esos trámites forman parte del mismo hecho administrativo.
- Si hay varios actos sobre el mismo hecho principal, inclúyelos en administrative_actions.
- event_summary debe ser breve, no nulo y en español.
- event_type debe describir la evolución material o administrativa principal: proyecto_nuevo, hibridacion, modificacion, repotenciacion, incorporacion_almacenamiento, proyecto_infraestructura_autonoma, cambio_titularidad, terminacion, otro o desconocido.

AdministrativeAction:
- administrative_actions debe recoger los actos administrativos publicados en el BOE.
- procedure_stage identifica el trámite o acto administrativo.
- procedure_decision identifica la decisión publicada.
- Extrae todos los actos administrativos que formen parte del objeto de publicación.
- No conviertas antecedentes históricos en administrative_actions del evento actual.
- Cada administrative_action debe tener evidence específica.

EnergyProjectMention:
- project_mentions debe recoger solo proyectos o instalaciones sustantivas afectadas por el hecho publicado.
- Extrae parques eólicos, plantas fotovoltaicas, almacenamiento, hibridaciones, repotenciaciones, modificaciones e infraestructuras autónomas cuando sean objeto principal.
- No extraigas líneas, SET, subestaciones, posiciones o evacuación auxiliar como EnergyProjectMention salvo que sean el objeto principal del BOE.
- Si una infraestructura es auxiliar del proyecto principal, resúmela en associated_infrastructure.
- No extraigas proyectos mencionados solo como contexto, antecedentes, agrupaciones, complejos energéticos, comparaciones o referencias informativas.
- Cada project_mention debe tener evidence específica.
- role_in_event debe indicar si la mención es objeto_principal, referencia_existente, referencia_asociada o desconocido.
- status_in_document solo refleja el estado mencionado o fuertemente implicado en esta publicación concreta, no el estado consolidado.

Technical attributes:
- technical_attributes recoge el tipo de instalación o tecnología y magnitudes técnicas explícitas.
- installation_type es una clasificación controlada, no una copia literal del BOE.
- installed_power_mw recoge potencia instalada o nominal en MW.
- peak_power_mwp recoge potencia pico en MWp.
- storage_power_mw recoge potencia del almacenamiento.
- storage_capacity_mwh recoge capacidad del almacenamiento.
- No transfieras potencias entre proyectos.
- No deduzcas potencias salvo que la equivalencia sea explícita.
- Explica conversiones o discrepancias en power_normalization_note.

Interpretación numérica:
- Interpreta números con formato español.
- "31,172 MW" = 31.172 MW.
- "28.000 kW" = 28000 kW.
- Si el texto incluye número de equipos y potencia unitaria, calcula la potencia total solo si la equivalencia es explícita.
- Si hay discrepancia entre cálculo y cifra textual, consérvala en power_normalization_note sin afirmar que el BOE contiene una errata.

AssociatedInfrastructureSummary:
- associated_infrastructure resume infraestructura auxiliar relevante y no debe ser exhaustiva.
- Usa has_evacuation_infrastructure = true si se menciona evacuación, red colectora, línea de evacuación o infraestructura común.
- Usa has_electrical_substation = true si se mencionan SET, SE o subestaciones.
- Usa has_grid_connection = true si se menciona punto de conexión, REE, red de transporte/distribución o posición de conexión.
- Usa has_shared_infrastructure = true si la infraestructura parece compartida, colectora o común.
- description debe resumir la infraestructura en español.
- evidence debe justificar el resumen.

Hibridación y almacenamiento:
- Si el evento principal es una hibridación, event_type = "hibridacion".
- Extrae la nueva instalación como objeto_principal.
- Extrae la instalación existente como referencia_existente si aparece explícitamente.
- No crees relaciones entre activos: el contrato no contiene asset_relations.
- Si el evento principal es añadir almacenamiento, event_type = "incorporacion_almacenamiento".
- El almacenamiento puede ser objeto principal, no solo infraestructura auxiliar.

Participantes:
- Extrae promotores, copromotores, titulares, operadores, gestores de red, órganos sustantivos, órganos ambientales y administraciones solo si aparecen explícitamente.
- Conserva la denominación literal.
- No inventes CIF, NIF ni identificadores societarios.
- No resuelvas entidades por conocimiento externo.

Localizaciones:
- Extrae solo municipios, provincias y comunidades autónomas explícitamente mencionados.
- Conserva la forma textual del BOE.
- No normalices nombres ni generes códigos INE.
- Usa province_hint y autonomous_community_hint solo si aparecen explícitamente o en contexto inmediato.
- Si no hay localización explícita para un project_mention, devuelve locations = [].

Notas:
- extraction_notes debe registrar incertidumbres relevantes.
"""

In [19]:
MAPPING_HINTS = """
Mapeos orientativos BOE -> valores enum:

procedure_stage:
- solicitud, solicitud de autorización -> solicitud_tramitacion
- solicitud de evaluación ambiental, solicitud de determinación de afección ambiental -> solicitud_tramitacion_ambiental
- información pública, se somete a información pública -> informacion_publica
- evaluación de impacto ambiental -> evaluacion_impacto_ambiental
- declaración de impacto ambiental, DIA -> declaracion_impacto_ambiental
- informe de impacto ambiental -> informe_impacto_ambiental
- informe de determinación de afección ambiental, IDAA -> informe_determinacion_afeccion_ambiental
- autorización administrativa previa, AAP -> autorizacion_administrativa_previa
- autorización administrativa de construcción, AAC -> autorizacion_administrativa_construccion
- autorización de explotación -> autorizacion_explotacion
- declaración de utilidad pública, DUP -> declaracion_utilidad_publica
- relación de bienes y derechos afectados -> relacion_bienes_derechos_afectados
- levantamiento de actas previas a la ocupación -> levantamiento_actas_previas_ocupacion
- actas de ocupación -> actas_ocupacion
- modificación -> modificacion
- prórroga -> prorroga
- archivo del expediente -> archivo_expediente
- desistimiento -> desistimiento
- inadmisión -> inadmision
- cambio de titularidad, transmisión de titularidad -> cambio_titularidad

procedure_decision:
- solicita, solicitud -> solicitado
- subsanación -> subsanado
- se formula -> formulado
- favorable -> favorable
- desfavorable -> desfavorable
- ambientalmente viable -> ambientalmente_viable
- ambientalmente no viable -> ambientalmente_no_viable
- debe someterse a evaluación ambiental adicional u ordinaria -> requiere_evaluacion_ambiental_adicional
- no debe someterse a evaluación ambiental adicional u ordinaria -> no_requiere_evaluacion_ambiental_adicional
- se somete a información pública -> sometido_informacion_publica
- se convoca -> convocado
- se declara de utilidad pública -> declarado_utilidad_publica
- se aprueba -> aprobado
- se autoriza, se otorga autorización -> autorizado
- se deniega -> denegado
- se archiva -> archivado
- desiste -> desistido
- se inadmite -> inadmitido

event_type:
- nueva planta, nuevo parque, nueva instalación -> proyecto_nuevo
- hibridación -> hibridacion
- modificación, ampliación, cambio de configuración -> modificacion
- repotenciación -> repotenciacion
- almacenamiento asociado, incorporación de baterías, BESS asociado -> incorporacion_almacenamiento
- línea, subestación o infraestructura autónoma como objeto principal -> proyecto_infraestructura_autonoma
- cambio/transmisión de titularidad -> cambio_titularidad
- archivo, desistimiento, inadmisión o denegación final -> terminacion

installation_type:
- planta solar, planta fotovoltaica, PFV, FV -> fotovoltaica
- parque eólico, PE, aerogeneradores -> eolica
- termosolar, solar termoeléctrica -> termosolar
- hidroeléctrica -> hidroelectrica
- biomasa -> biomasa
- biogás -> biogas
- hidrógeno verde, hidrógeno renovable -> hidrogeno_verde
- BESS, baterías, almacenamiento -> almacenamiento
- línea eléctrica, línea aérea, línea subterránea, LAT, LSAT -> linea_electrica
- subestación, SET, SE -> subestacion_electrica
- evacuación, infraestructura de evacuación, infraestructura común de evacuación -> infraestructura_evacuacion
"""

In [20]:
### Agent instructions

AGENT_INSTRUCTIONS = "\n\n".join(
    [
        INSTRUCTIONS.strip(),
        MAPPING_HINTS.strip(),
    ]
)

## 3. Agente

### Build Agent

In [21]:
def build_agent(
    model,
    output_type: type[BaseModel],
    instructions: str,
    *,
    retries: int = 3,
) -> Agent:
    """
    Build a Pydantic AI agent for structured BOE extraction.

    Parameters
    ----------
    model
        Model identifier or model instance accepted by Pydantic AI.
    output_type
        Pydantic model defining the expected structured output.
    instructions
        Full extraction instructions passed to the model.
    retries
        Number of retries for structured output validation.

    Returns
    -------
    Agent
        Configured Pydantic AI agent.
    """
    return Agent(
        model,
        output_type=output_type,
        instructions=instructions,
        retries=retries,
    )

In [22]:
# Ollama model builder

from pydantic_ai.models.ollama import OllamaModel
from pydantic_ai.providers.ollama import OllamaProvider


def build_ollama_model(
    model_name: str,
    *,
    base_url: str = "http://localhost:11434/v1",
) -> OllamaModel:
    """
    Build an Ollama model instance compatible with Pydantic AI.

    Parameters
    ----------
    model_name
        Local Ollama model name, for example "qwen3:8b".
    base_url
        Ollama OpenAI-compatible endpoint.

    Returns
    -------
    OllamaModel
        Configured Ollama model.
    """
    return OllamaModel(
        model_name,
        provider=OllamaProvider(base_url=base_url),
    )

### Modelos disponibles

In [23]:
MODEL_PROVIDER = "gemini"
# MODEL_PROVIDER = "ollama"


if MODEL_PROVIDER == "gemini":
    AI_MODEL_NAME = "google:gemini-2.5-flash"
    AI_MODEL = AI_MODEL_NAME
    AGENT_RETRIES = 3

elif MODEL_PROVIDER == "ollama":
    AI_MODEL_NAME = "qwen3:8b"
    AI_MODEL = build_ollama_model(AI_MODEL_NAME)
    AGENT_RETRIES = 4  # Local models usually need slightly more validation retries.

else:
    raise ValueError(
        f"Unsupported model provider: {MODEL_PROVIDER}"
    )

In [24]:
agent = build_agent(
    model=AI_MODEL,
    output_type=BOEProjectExtraction,
    instructions=AGENT_INSTRUCTIONS,
    retries=AGENT_RETRIES,
)

## 4. Extracción con IA

In [25]:
TEXT_LIMIT = 4000

AI_EXTRACTION_LOG_COLUMNS = [
    "identificador_boe",
    "boe_id_extracted",
    "fecha_publicacion",
    "publication_date_extracted",
    "titulo",
    "energy_relevance",
    "is_project_specific",
    "n_publication_events",
    "extraction_json",
    "extracted_at",
    "model_name",
    "extraction_status",
    "error_type",
    "parse_error",
]

### Funciones

#### Utilidades internas

In [26]:
def empty_ai_extractions_log() -> pd.DataFrame:
    """
    Crea un log vacío de extracciones IA.

    Se usa cuando todavía no existe el fichero Parquet acumulado de
    extracciones. Devuelve un DataFrame con las columnas esperadas, pero sin
    registros.
    """
    return pd.DataFrame(columns=AI_EXTRACTION_LOG_COLUMNS)

In [27]:
def normalise_ai_extractions_log(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Normaliza la estructura tabular del log de extracciones IA.

    Garantiza que existan todas las columnas definidas en
    `AI_EXTRACTION_LOG_COLUMNS`, conserva posibles columnas adicionales al final
    y normaliza los tipos mínimos necesarios para trabajar de forma estable.

    Esta función no valida el contenido semántico del JSON extraído. Solo
    prepara el DataFrame del log.
    """
    ai_extractions = ai_extractions.copy()

    for col in AI_EXTRACTION_LOG_COLUMNS:
        if col not in ai_extractions.columns:
            ai_extractions[col] = pd.NA

    extra_cols = [
        col
        for col in ai_extractions.columns
        if col not in AI_EXTRACTION_LOG_COLUMNS
    ]

    ai_extractions = ai_extractions[
        AI_EXTRACTION_LOG_COLUMNS + extra_cols
    ]

    if ai_extractions.empty:
        return ai_extractions

    ai_extractions["identificador_boe"] = (
        ai_extractions["identificador_boe"]
        .astype("string")
    )

    ai_extractions["boe_id_extracted"] = (
        ai_extractions["boe_id_extracted"]
        .astype("string")
    )

    ai_extractions["fecha_publicacion"] = pd.to_datetime(
        ai_extractions["fecha_publicacion"],
        errors="coerce",
    )

    ai_extractions["publication_date_extracted"] = pd.to_datetime(
        ai_extractions["publication_date_extracted"],
        errors="coerce",
    )

    return ai_extractions


In [28]:
def validate_extraction_metadata(
    row: pd.Series,
    extraction: BOEProjectExtraction,
) -> None:
    """
    Valida la coherencia entre el documento fuente y la extracción IA.

    El identificador BOE y la fecha de publicación proceden del dataset fuente
    y se consideran metadatos canónicos. La IA puede devolver esos mismos
    campos dentro del JSON, pero no debe contradecirlos.

    La validación falla si:
    - `extraction.boe_id` no coincide con `row["identificador"]`.
    - `extraction.publication_date` y `row["fecha_publicacion"]` existen y son
      fechas distintas.

    No falla si una de las dos fechas es nula, porque en ese caso no hay
    contradicción verificable.
    """
    source_boe_id = safe_str(row["identificador"])
    extracted_boe_id = safe_str(extraction.boe_id)

    if source_boe_id != extracted_boe_id:
        raise ValueError(
            "Identificador BOE incoherente: "
            f"fuente={source_boe_id!r}, "
            f"extraccion={extracted_boe_id!r}"
        )

    source_date = to_date_or_none(row["fecha_publicacion"])
    extracted_date = to_date_or_none(extraction.publication_date)

    if (
        source_date is not None
        and extracted_date is not None
        and source_date != extracted_date
    ):
        raise ValueError(
            "Fecha de publicación incoherente: "
            f"fuente={source_date}, "
            f"extraccion={extracted_date}"
        )

#### Construcción de prompt

In [29]:
def build_prompt(
    row: pd.Series,
    text_limit: int = TEXT_LIMIT,
) -> str:
    """
    Construye el prompt documental enviado al agente de extracción.
    """
    texto_limpio = safe_str(row["texto_limpio"])

    return f"""
Identificador BOE: {row["identificador"]}
Fecha publicación: {row["fecha_publicacion"]}
Título: {row["titulo"]}

Texto:
{texto_limpio[:text_limit]}
""".strip()

#### Registro de extracción

In [30]:
def build_ai_extraction_record(
    row: pd.Series,
    extraction: BOEProjectExtraction,
    model_name: str,
) -> dict:
    """
    Construye un registro tabular a partir de una extracción IA validada.

    La clave documental canónica se toma del dataset fuente. Los valores
    devueltos por la IA se conservan también para auditoría.
    """
    validate_extraction_metadata(
        row=row,
        extraction=extraction,
    )

    return {
        "identificador_boe": row["identificador"],
        "boe_id_extracted": extraction.boe_id,
        "fecha_publicacion": pd.to_datetime(
            row["fecha_publicacion"],
            errors="coerce",
        ),
        "publication_date_extracted": pd.to_datetime(
            extraction.publication_date,
            errors="coerce",
        ),
        "titulo": row["titulo"],
        "energy_relevance": enum_value(extraction.energy_relevance),
        "is_project_specific": extraction.is_project_specific,
        "n_publication_events": len(extraction.publication_events),
        "extraction_json": extraction.model_dump_json(),
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "ok",
        "error_type": None,
        "parse_error": None,
    }

In [31]:
def build_ai_error_record(
    row: pd.Series,
    model_name: str,
    error: Exception,
) -> dict:
    """
    Construye un registro tabular cuando falla la extracción IA.
    """
    return {
        "identificador_boe": row["identificador"],
        "boe_id_extracted": None,
        "fecha_publicacion": pd.to_datetime(
            row["fecha_publicacion"],
            errors="coerce",
        ),
        "publication_date_extracted": pd.NaT,
        "titulo": row["titulo"],
        "energy_relevance": None,
        "is_project_specific": None,
        "n_publication_events": None,
        "extraction_json": None,
        "extracted_at": datetime.now(timezone.utc).isoformat(),
        "model_name": model_name,
        "extraction_status": "error",
        "error_type": type(error).__name__,
        "parse_error": str(error),
    }

#### Carga y upsert incremental

In [32]:
def load_ai_extractions(
    output_path: Path,
) -> pd.DataFrame:
    """
    Carga el log acumulado de extracciones IA.

    Si el fichero no existe, devuelve un DataFrame vacío con las columnas
    esperadas.
    """
    if not output_path.exists():
        return empty_ai_extractions_log()

    ai_extractions = pd.read_parquet(output_path)

    return normalise_ai_extractions_log(ai_extractions)

In [33]:
def upsert_ai_extractions(
    new_ai_extractions: pd.DataFrame,
    output_path: Path,
) -> pd.DataFrame:
    """
    Inserta o actualiza extracciones IA en un Parquet acumulado.

    Conserva la última extracción de cada `identificador_boe`.

    Esta función debe usarse solo para la tabla fuente de extracciones IA. Las
    tablas derivadas deben regenerarse desde esta tabla.
    """
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    existing_ai_extractions = load_ai_extractions(output_path)

    new_ai_extractions = normalise_ai_extractions_log(
        new_ai_extractions
    )

    if new_ai_extractions.empty:
        return existing_ai_extractions

    ai_extractions = pd.concat(
        [
            existing_ai_extractions,
            new_ai_extractions,
        ],
        ignore_index=True,
    )

    ai_extractions = ai_extractions.drop_duplicates(
        subset=["identificador_boe"],
        keep="last",
    )

    ai_extractions = normalise_ai_extractions_log(
        ai_extractions
    )

    save_parquet(
        ai_extractions,
        output_path,
    )

    return ai_extractions

### Cargar candidatos BOE

In [34]:
df = pd.read_parquet(BOE_CANDIDATES_DOCS_TEXT_PATH)

required_input_cols = {
    "identificador",
    "fecha_publicacion",
    "titulo",
    "xml_status",
    "texto_limpio",
}

validate_required_columns(
    df,
    required_input_cols,
)

df = df.loc[
    (df["xml_status"] == "ok")
    & df["texto_limpio"].notna()
    & (df["texto_limpio"].astype(str).str.len() > 0)
].copy()

df["identificador"] = df["identificador"].astype("string")

print(f"{len(df)=}")

len(df)=1266


### Filtrar BOEs ya procesados correctamente

In [35]:
ai_extractions = load_ai_extractions(
    BOE_AI_EXTRACTIONS_PATH
)

processed_ids = set(
    ai_extractions.loc[
        ai_extractions["extraction_status"] == "ok",
        "identificador_boe",
    ]
    .dropna()
    .astype(str)
)

pending_df = df.loc[
    ~df["identificador"].isin(processed_ids)
].copy()

print(f"{len(processed_ids)=}")
print(f"{len(pending_df)=}")

len(processed_ids)=0
len(pending_df)=1266


### Seleccionar proyectos de test

#### PE Badulaque

In [36]:
target_ids_badulaque = [
    "BOE-B-2021-32560",
    "BOE-A-2023-2598",
    "BOE-A-2023-10306",
    "BOE-B-2023-19082",
    "BOE-A-2024-16664",
]

#### FV Andévalo

Buenos casos de prueba para comprobar que el sistema no agrupa proyectos distintos simplemente porque comparten infraestructura de evacuación o mencionan FV Andévalo.

In [37]:
target_ids_andevalo = [
    # Proyecto FV Andévalo e hibridaciones
    "BOE-B-2024-26379",
    "BOE-A-2025-18285",
    "BOE-B-2026-3596",

    # Antecedentes y referencias indirectas
    "BOE-A-2022-24404",  # FV Majal Alto
    "BOE-A-2024-9608",   # FV La Puebla 1
    "BOE-A-2025-26110",  # FV La Puebla 1
    "BOE-A-2026-7629",   # FV La Puebla 3 y 4
    "BOE-A-2026-13454",  # FV La Puebla 3
]

In [38]:
target_ids = target_ids_badulaque + target_ids_andevalo

In [39]:
missing_target_ids = sorted(
    set(target_ids) - set(df["identificador"].astype(str))
)

if missing_target_ids:
    print("target_ids no encontrados en `df`:")
    print(missing_target_ids)

df_test_initial = df.loc[
    df["identificador"].isin(target_ids)
].copy()

df_test_pending = df_test_initial.loc[
    ~df_test_initial["identificador"].isin(processed_ids)
].copy()

print(f"{len(df_test_initial)=}")
print(f"{len(df_test_pending)=}")

len(df_test_initial)=13
len(df_test_pending)=13


In [40]:
display(
    df_test_initial[
        [
            "identificador",
            "fecha_publicacion",
            "titulo",
        ]
    ].sort_values("identificador")
)

,identificador,fecha_publicacion,titulo
16,BOE-A-2022-24404,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D..."
98,BOE-A-2023-10306,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc..."
76,BOE-A-2023-2598,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc..."
213,BOE-A-2024-16664,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc..."
142,BOE-A-2024-9608,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci..."
235,BOE-A-2025-18285,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc..."
275,BOE-A-2025-26110,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D..."
1264,BOE-A-2026-13454,2026-06-20,"Resolución de 3 de junio de 2026, de la Direcc..."
310,BOE-A-2026-7629,2026-04-03,"Resolución de 17 de marzo de 2026, de la Direc..."
6,BOE-B-2021-32560,2021-07-07,Anuncio del Área de Industria y Energía de la ...


## 5. Extracción con agente sobre los pendientes

In [41]:
## 5. Extracción con agente sobre los pendientes

# Modo de ejecución.
# Durante el desarrollo se usan solo los casos de prueba pendientes
# para validar el contrato y revisar errores de forma controlada.
run_df = df_test_pending.copy()

# TODO: Para procesar todos los BOE pendientes, sustituir la línea anterior por:
# run_df = pending_df.copy()


# Lista donde se acumula un registro tabular por cada documento procesado.
# Cada registro será una fila del log de extracciones IA.
records = []


# Variables auxiliares de depuración.
# No se guardan en parquet. Solo permiten inspeccionar en el notebook
# el último prompt, resultado, extracción o error producido.
last_prompt = None
last_result = None
last_extraction = None
last_error = None


print(f"{len(run_df)=}")


for i, (_, row) in enumerate(run_df.iterrows(), start=1):
    boe_id = row["identificador"]

    print(f"[{i}/{len(run_df)}] Extrayendo {boe_id}")

    # Construye el prompt documental a partir de la fila BOE.
    # Incluye identificador, fecha, título y texto limpio recortado.
    last_prompt = build_prompt(row)

    try:
        # Ejecuta el agente IA y valida la salida contra BOEProjectExtraction.
        last_result = await agent.run(last_prompt)
        last_extraction = last_result.output
        last_error = None

        # Convierte la extracción Pydantic validada en una fila tabular.
        # Aquí también se valida que boe_id y publication_date no contradigan
        # los metadatos canónicos del documento fuente.
        record = build_ai_extraction_record(
            row=row,
            extraction=last_extraction,
            model_name=AI_MODEL_NAME,
        )

    except Exception as exc:
        # Si falla la llamada al modelo, la validación Pydantic o la validación
        # de metadatos, se registra una fila de error en lugar de interrumpir
        # todo el procesamiento.
        last_error = exc

        record = build_ai_error_record(
            row=row,
            model_name=AI_MODEL_NAME,
            error=exc,
        )

        print(f"  ERROR {type(exc).__name__}: {exc}")

    records.append(record)


# Convierte los registros nuevos en DataFrame.
new_ai_extractions = pd.DataFrame(records)

# Normaliza columnas y tipos mínimos del log antes de mostrar o guardar.
# Esto garantiza que las extracciones correctas y los errores tengan
# la misma estructura tabular.
new_ai_extractions = normalise_ai_extractions_log(
    new_ai_extractions
)

display(new_ai_extractions)

len(run_df)=13
[1/13] Extrayendo BOE-B-2021-32560
[2/13] Extrayendo BOE-A-2022-24404
[3/13] Extrayendo BOE-A-2023-2598
[4/13] Extrayendo BOE-A-2023-10306
[5/13] Extrayendo BOE-B-2023-19082
[6/13] Extrayendo BOE-A-2024-9608
[7/13] Extrayendo BOE-B-2024-26379
[8/13] Extrayendo BOE-A-2024-16664
[9/13] Extrayendo BOE-A-2025-18285
[10/13] Extrayendo BOE-A-2025-26110
[11/13] Extrayendo BOE-B-2026-3596
[12/13] Extrayendo BOE-A-2026-7629
[13/13] Extrayendo BOE-A-2026-13454


,identificador_boe,boe_id_extracted,fecha_publicacion,publication_date_extracted,titulo,energy_relevance,is_project_specific,n_publication_events,extraction_json,extracted_at,model_name,extraction_status,error_type,parse_error
0,BOE-B-2021-32560,BOE-B-2021-32560,2021-07-07,2021-07-07,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2021-32560"",""publication_date...",2026-07-10T06:38:36.624039+00:00,google:gemini-2.5-flash,ok,None,None
1,BOE-A-2022-24404,BOE-A-2022-24404,2022-12-30,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2022-24404"",""publication_date...",2026-07-10T06:38:52.920954+00:00,google:gemini-2.5-flash,ok,None,None
2,BOE-A-2023-2598,BOE-A-2023-2598,2023-01-31,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-2598"",""publication_date""...",2026-07-10T06:39:06.462705+00:00,google:gemini-2.5-flash,ok,None,None
3,BOE-A-2023-10306,BOE-A-2023-10306,2023-04-28,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-10306"",""publication_date...",2026-07-10T06:39:19.870232+00:00,google:gemini-2.5-flash,ok,None,None
4,BOE-B-2023-19082,BOE-B-2023-19082,2023-06-22,2023-06-22,Anuncio del Área Funcional de Industria y Ener...,relevante,True,1,"{""boe_id"":""BOE-B-2023-19082"",""publication_date...",2026-07-10T06:39:37.533247+00:00,google:gemini-2.5-flash,ok,None,None
5,BOE-A-2024-9608,BOE-A-2024-9608,2024-05-13,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci...",relevante,True,1,"{""boe_id"":""BOE-A-2024-9608"",""publication_date""...",2026-07-10T06:39:46.411786+00:00,google:gemini-2.5-flash,ok,None,None
6,BOE-B-2024-26379,BOE-B-2024-26379,2024-07-13,2024-07-13,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2024-26379"",""publication_date...",2026-07-10T06:40:08.864930+00:00,google:gemini-2.5-flash,ok,None,None
7,BOE-A-2024-16664,BOE-A-2024-16664,2024-08-10,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2024-16664"",""publication_date...",2026-07-10T06:40:21.417262+00:00,google:gemini-2.5-flash,ok,None,None
8,BOE-A-2025-18285,BOE-A-2025-18285,2025-09-15,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2025-18285"",""publication_date...",2026-07-10T06:40:32.063020+00:00,google:gemini-2.5-flash,ok,None,None
9,BOE-A-2025-26110,BOE-A-2025-26110,2025-12-19,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2025-26110"",""publication_date...",2026-07-10T06:40:44.417571+00:00,google:gemini-2.5-flash,ok,None,None


In [42]:
# Inserta o actualiza el log acumulado de extracciones IA.
# Si un identificador BOE ya existía, se conserva la última versión.
ai_extractions = upsert_ai_extractions(
    new_ai_extractions=new_ai_extractions,
    output_path=BOE_AI_EXTRACTIONS_PATH,
)

display(ai_extractions)

/tmp/ipykernel_643879/2444363489.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ai_extractions = pd.concat(


,identificador_boe,boe_id_extracted,fecha_publicacion,publication_date_extracted,titulo,energy_relevance,is_project_specific,n_publication_events,extraction_json,extracted_at,model_name,extraction_status,error_type,parse_error
0,BOE-B-2021-32560,BOE-B-2021-32560,2021-07-07,2021-07-07,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2021-32560"",""publication_date...",2026-07-10T06:38:36.624039+00:00,google:gemini-2.5-flash,ok,None,None
1,BOE-A-2022-24404,BOE-A-2022-24404,2022-12-30,2022-12-30,"Resolución de 22 de diciembre de 2022, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2022-24404"",""publication_date...",2026-07-10T06:38:52.920954+00:00,google:gemini-2.5-flash,ok,None,None
2,BOE-A-2023-2598,BOE-A-2023-2598,2023-01-31,2023-01-31,"Resolución de 23 de enero de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-2598"",""publication_date""...",2026-07-10T06:39:06.462705+00:00,google:gemini-2.5-flash,ok,None,None
3,BOE-A-2023-10306,BOE-A-2023-10306,2023-04-28,2023-04-28,"Resolución de 17 de abril de 2023, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2023-10306"",""publication_date...",2026-07-10T06:39:19.870232+00:00,google:gemini-2.5-flash,ok,None,None
4,BOE-B-2023-19082,BOE-B-2023-19082,2023-06-22,2023-06-22,Anuncio del Área Funcional de Industria y Ener...,relevante,True,1,"{""boe_id"":""BOE-B-2023-19082"",""publication_date...",2026-07-10T06:39:37.533247+00:00,google:gemini-2.5-flash,ok,None,None
5,BOE-A-2024-9608,BOE-A-2024-9608,2024-05-13,2024-05-13,"Resolución de 6 de mayo de 2024, de la Direcci...",relevante,True,1,"{""boe_id"":""BOE-A-2024-9608"",""publication_date""...",2026-07-10T06:39:46.411786+00:00,google:gemini-2.5-flash,ok,None,None
6,BOE-B-2024-26379,BOE-B-2024-26379,2024-07-13,2024-07-13,Anuncio del Área de Industria y Energía de la ...,relevante,True,1,"{""boe_id"":""BOE-B-2024-26379"",""publication_date...",2026-07-10T06:40:08.864930+00:00,google:gemini-2.5-flash,ok,None,None
7,BOE-A-2024-16664,BOE-A-2024-16664,2024-08-10,2024-08-10,"Resolución de 22 de julio de 2024, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2024-16664"",""publication_date...",2026-07-10T06:40:21.417262+00:00,google:gemini-2.5-flash,ok,None,None
8,BOE-A-2025-18285,BOE-A-2025-18285,2025-09-15,2025-09-15,"Resolución de 7 de agosto de 2025, de la Direc...",relevante,True,1,"{""boe_id"":""BOE-A-2025-18285"",""publication_date...",2026-07-10T06:40:32.063020+00:00,google:gemini-2.5-flash,ok,None,None
9,BOE-A-2025-26110,BOE-A-2025-26110,2025-12-19,2025-12-19,"Resolución de 17 de noviembre de 2025, de la D...",relevante,True,1,"{""boe_id"":""BOE-A-2025-26110"",""publication_date...",2026-07-10T06:40:44.417571+00:00,google:gemini-2.5-flash,ok,None,None


In [43]:
# Resumen global del estado del log acumulado.
# Permite comprobar rápidamente cuántas extracciones han terminado en "ok"
# y cuántas han quedado como "error".
if not ai_extractions.empty:
    display(
        ai_extractions["extraction_status"]
        .value_counts(dropna=False)
        .rename_axis("extraction_status")
        .reset_index(name="n")
    )

,extraction_status,n
0,ok,13


In [44]:
# Vista compacta de los registros generados en esta ejecución.
# Sirve para revisar sin abrir el JSON completo:
# - relevancia energética,
# - si el documento es específico de proyecto,
# - número de eventos extraídos,
# - estado de extracción,
# - error, si lo hubo.
if not new_ai_extractions.empty:
    display(
        new_ai_extractions[
            [
                "identificador_boe",
                "fecha_publicacion",
                "energy_relevance",
                "is_project_specific",
                "n_publication_events",
                "extraction_status",
                "error_type",
                "parse_error",
            ]
        ]
    )

,identificador_boe,fecha_publicacion,energy_relevance,is_project_specific,n_publication_events,extraction_status,error_type,parse_error
0,BOE-B-2021-32560,2021-07-07,relevante,True,1,ok,None,None
1,BOE-A-2022-24404,2022-12-30,relevante,True,1,ok,None,None
2,BOE-A-2023-2598,2023-01-31,relevante,True,1,ok,None,None
3,BOE-A-2023-10306,2023-04-28,relevante,True,1,ok,None,None
4,BOE-B-2023-19082,2023-06-22,relevante,True,1,ok,None,None
5,BOE-A-2024-9608,2024-05-13,relevante,True,1,ok,None,None
6,BOE-B-2024-26379,2024-07-13,relevante,True,1,ok,None,None
7,BOE-A-2024-16664,2024-08-10,relevante,True,1,ok,None,None
8,BOE-A-2025-18285,2025-09-15,relevante,True,1,ok,None,None
9,BOE-A-2025-26110,2025-12-19,relevante,True,1,ok,None,None


## 6. Chequeo de lo extraído

In [45]:
extracted = load_ai_extractions(
    BOE_AI_EXTRACTIONS_PATH
)

In [46]:
def show_extraction_json(
    ai_extractions: pd.DataFrame,
    boe_id: str,
) -> None:
    """
    Muestra el JSON extraído para un identificador BOE.

    Usa la última extracción disponible para ese identificador y comprueba que
    esté en estado `ok`.
    """
    mask = ai_extractions["identificador_boe"].eq(boe_id)

    if not mask.any():
        raise ValueError(
            f"No hay extracción registrada para {boe_id}"
        )

    rows = ai_extractions.loc[mask].copy()

    rows["extracted_at"] = pd.to_datetime(
        rows["extracted_at"],
        errors="coerce",
    )

    row = (
        rows.sort_values("extracted_at")
        .iloc[-1]
    )

    if row["extraction_status"] != "ok":
        raise ValueError(
            f"La extracción de {boe_id} no está en estado ok. "
            f"Estado: {row['extraction_status']}. "
            f"Error: {row['parse_error']}"
        )

    if pd.isna(row["extraction_json"]):
        raise ValueError(
            f"La extracción de {boe_id} no contiene JSON."
        )

    try:
        parsed_json = json.loads(row["extraction_json"])
    except json.JSONDecodeError as exc:
        raise ValueError(
            f"El JSON extraído para {boe_id} no es válido."
        ) from exc

    print(
        json.dumps(
            parsed_json,
            indent=2,
            ensure_ascii=False,
        )
    )

In [47]:
boe_id_to_check = "BOE-B-2021-32560"

show_extraction_json(
    extracted,
    boe_id_to_check,
)

{
  "boe_id": "BOE-B-2021-32560",
  "publication_date": "2021-07-07",
  "energy_relevance": "relevante",
  "is_project_specific": true,
  "relevance_reason": "El documento somete a información pública un proyecto de parque eólico específico (Badulaque) y su infraestructura de evacuación.",
  "publication_events": [
    {
      "event_type": "proyecto_nuevo",
      "administrative_actions": [
        {
          "procedure_stage": "autorizacion_administrativa_previa",
          "procedure_decision": "solicitado",
          "evidence": "solicitud de Autorización Administrativa Previa"
        },
        {
          "procedure_stage": "informacion_publica",
          "procedure_decision": "sometido_informacion_publica",
          "evidence": "se somete al trámite de información pública, de forma conjunta, el Estudio de Impacto Ambiental y la solicitud de Autorización Administrativa Previa"
        }
      ],
      "project_mentions": [
        {
          "local_project_id": "project_1",


## 7. Flattening data in field `extraction_json`

### Funciones

#### Columnas esperadas

In [48]:
# Las columnas esperadas sirven para que cada función devuelva siempre un DataFrame con la misma estructura, incluso cuando no hay registros.
# El objetivo es tener un pipeline más determinista.

PUBLICATION_EVENTS_COLUMNS = [
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "event_index",
    "event_type",
    "event_summary",
    "evidence",
]

ADMINISTRATIVE_ACTIONS_COLUMNS = [
    "action_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "action_index",
    "procedure_stage",
    "procedure_decision",
    "evidence",
]

PROJECT_MENTIONS_COLUMNS = [
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "local_project_id",
    "project_name",
    "project_name_norm",
    "role_in_event",
    "status_in_document",
    "case_file_reference",
    "evidence",
]

PROJECT_TECHNICAL_ATTRIBUTES_COLUMNS = [
    "project_technical_attribute_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "attribute_index",
    "installation_type",
    "installed_power_mw",
    "peak_power_mwp",
    "storage_power_mw",
    "storage_capacity_mwh",
    "description",
    "power_normalization_note",
    "evidence",
]

PROJECT_PARTICIPANTS_COLUMNS = [
    "project_participant_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "participant_index",
    "participant_name",
    "participant_name_norm",
    "participant_role",
    "evidence",
]

PROJECT_LOCATIONS_COLUMNS = [
    "project_location_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "location_index",
    "municipality_raw",
    "municipality_raw_norm",
    "province_hint_raw",
    "province_hint_raw_norm",
    "autonomous_community_hint_raw",
    "autonomous_community_hint_raw_norm",
    "location_evidence",
]

PROJECT_ALIASES_COLUMNS = [
    "project_alias_id",
    "project_mention_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "alias_index",
    "alias",
    "alias_norm",
]

ASSOCIATED_INFRASTRUCTURE_COLUMNS = [
    "associated_infrastructure_id",
    "event_id",
    "identificador_boe",
    "fecha_publicacion",
    "has_evacuation_infrastructure",
    "has_electrical_substation",
    "has_grid_connection",
    "has_shared_infrastructure",
    "description",
    "evidence",
]

#### Utilidades de flattening

In [49]:
INVALID_LOCATION_VALUES = {
    "",
    "no consta",
    "desconocido",
    "no aplica",
    "ninguno",
}


def empty_df(columns: list[str]) -> pd.DataFrame:
    """
    Devuelve un DataFrame vacío con columnas estables.

    Evita que una tabla derivada quede sin columnas cuando no hay registros.
    """
    return pd.DataFrame(columns=columns)


def iter_valid_extractions(
    ai_extractions: pd.DataFrame,
):
    """
    Itera sobre extracciones IA válidas.

    Solo procesa filas con `extraction_status == "ok"` y `extraction_json`
    no nulo. Cada JSON se valida de nuevo contra `BOEProjectExtraction`.
    """
    required_cols = {
        "extraction_status",
        "extraction_json",
    }

    validate_required_columns(
        ai_extractions,
        required_cols,
    )

    valid_rows = ai_extractions.loc[
        (ai_extractions["extraction_status"] == "ok")
        & ai_extractions["extraction_json"].notna()
    ]

    for _, row in valid_rows.iterrows():
        yield BOEProjectExtraction.model_validate_json(
            row["extraction_json"]
        )


def make_event_id(
    boe_id: str,
    event_idx: int,
) -> str:
    """
    Construye un identificador estable de evento publicado.
    """
    return f"{boe_id}_event_{event_idx}"


def make_project_mention_id(
    event_id: str,
    local_project_id: str,
) -> str:
    """
    Construye un identificador estable de mención de proyecto.
    """
    return f"{event_id}_{local_project_id}"

#### flatten_publication_events

In [50]:
def flatten_publication_events(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana `publication_events`.

    Una fila representa un evento administrativo publicado en un documento BOE.
    No representa el ciclo de vida consolidado del proyecto.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            records.append(
                {
                    "event_id": event_id,
                    "identificador_boe": extraction.boe_id,
                    "fecha_publicacion": extraction.publication_date,
                    "event_index": event_idx,
                    "event_type": enum_value(event.event_type),
                    "event_summary": event.event_summary,
                    "evidence": event.evidence,
                }
            )

    if not records:
        return empty_df(PUBLICATION_EVENTS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PUBLICATION_EVENTS_COLUMNS,
    )

#### flatten_administrative_actions

In [51]:
def flatten_administrative_actions(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana los actos administrativos publicados.

    Una fila representa un trámite, acto o decisión administrativa dentro de un
    `PublicationEvent`.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for action_idx, action in enumerate(
                event.administrative_actions,
                start=1,
            ):
                action_id = (
                    f"{event_id}"
                    f"_action_{action_idx}"
                )

                records.append(
                    {
                        "action_id": action_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.boe_id,
                        "fecha_publicacion": extraction.publication_date,
                        "action_index": action_idx,
                        "procedure_stage": enum_value(action.procedure_stage),
                        "procedure_decision": enum_value(action.procedure_decision),
                        "evidence": action.evidence,
                    }
                )

    if not records:
        return empty_df(ADMINISTRATIVE_ACTIONS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=ADMINISTRATIVE_ACTIONS_COLUMNS,
    )

#### flatten_project_mentions

In [52]:
def flatten_project_mentions(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana las menciones de proyectos o instalaciones energéticas.

    Una fila representa una mención sustantiva de proyecto dentro de un evento
    publicado. Todavía no es un proyecto consolidado.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                records.append(
                    {
                        "project_mention_id": project_mention_id,
                        "event_id": event_id,
                        "identificador_boe": extraction.boe_id,
                        "fecha_publicacion": extraction.publication_date,
                        "local_project_id": project.local_project_id,
                        "project_name": project.name,
                        "project_name_norm": normalize_text(project.name),
                        "role_in_event": enum_value(project.role_in_event),
                        "status_in_document": enum_value(project.status_in_document),
                        "case_file_reference": project.case_file_reference,
                        "evidence": project.evidence,
                    }
                )

    if not records:
        return empty_df(PROJECT_MENTIONS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_MENTIONS_COLUMNS,
    )

#### flatten_project_technical_attributes

In [53]:
def flatten_project_technical_attributes(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana las características técnicas de cada mención de proyecto.

    Una fila representa un bloque técnico: tipo de instalación, potencia,
    almacenamiento o descripción técnica respaldada por evidencia.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for attribute_idx, attr in enumerate(
                    project.technical_attributes,
                    start=1,
                ):
                    records.append(
                        {
                            "project_technical_attribute_id": (
                                f"{project_mention_id}"
                                f"_attribute_{attribute_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": extraction.publication_date,
                            "attribute_index": attribute_idx,
                            "installation_type": enum_value(attr.installation_type),
                            "installed_power_mw": attr.installed_power_mw,
                            "peak_power_mwp": attr.peak_power_mwp,
                            "storage_power_mw": attr.storage_power_mw,
                            "storage_capacity_mwh": attr.storage_capacity_mwh,
                            "description": attr.description,
                            "power_normalization_note": attr.power_normalization_note,
                            "evidence": attr.evidence,
                        }
                    )

    if not records:
        return empty_df(PROJECT_TECHNICAL_ATTRIBUTES_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_TECHNICAL_ATTRIBUTES_COLUMNS,
    )

#### flatten_project_participants

In [54]:
def flatten_project_participants(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana participantes asociados a cada mención de proyecto.

    Incluye promotores, titulares, órganos administrativos u otras entidades
    extraídas explícitamente del BOE.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for participant_idx, participant in enumerate(
                    project.participants,
                    start=1,
                ):
                    records.append(
                        {
                            "project_participant_id": (
                                f"{project_mention_id}"
                                f"_participant_{participant_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": extraction.publication_date,
                            "participant_index": participant_idx,
                            "participant_name": participant.name,
                            "participant_name_norm": normalize_text(participant.name),
                            "participant_role": enum_value(participant.role),
                            "evidence": participant.evidence,
                        }
                    )

    if not records:
        return empty_df(PROJECT_PARTICIPANTS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_PARTICIPANTS_COLUMNS,
    )

#### flatten_project_locations

In [55]:
def flatten_project_locations(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana menciones textuales de localización.

    La IA solo extrae candidatos textuales. La resolución INE debe hacerse
    después mediante un proceso determinista.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for location_idx, loc in enumerate(
                    project.locations,
                    start=1,
                ):
                    municipality_raw = loc.municipality_name
                    municipality_raw_norm = normalize_text(municipality_raw)

                    if municipality_raw_norm in INVALID_LOCATION_VALUES:
                        continue

                    records.append(
                        {
                            "project_location_id": (
                                f"{project_mention_id}"
                                f"_location_{location_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": extraction.publication_date,
                            "location_index": location_idx,
                            "municipality_raw": municipality_raw,
                            "municipality_raw_norm": municipality_raw_norm,
                            "province_hint_raw": loc.province_hint,
                            "province_hint_raw_norm": normalize_text(loc.province_hint),
                            "autonomous_community_hint_raw": (
                                loc.autonomous_community_hint
                            ),
                            "autonomous_community_hint_raw_norm": normalize_text(
                                loc.autonomous_community_hint
                            ),
                            "location_evidence": loc.evidence,
                        }
                    )

    if not records:
        return empty_df(PROJECT_LOCATIONS_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_LOCATIONS_COLUMNS,
    )

#### flatten_project_aliases

In [56]:
def flatten_project_aliases(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana alias o denominaciones alternativas de cada mención de proyecto.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            for project in event.project_mentions:
                project_mention_id = make_project_mention_id(
                    event_id,
                    project.local_project_id,
                )

                for alias_idx, alias in enumerate(
                    project.aliases,
                    start=1,
                ):
                    alias_norm = normalize_text(alias)

                    if not alias_norm:
                        continue

                    records.append(
                        {
                            "project_alias_id": (
                                f"{project_mention_id}"
                                f"_alias_{alias_idx}"
                            ),
                            "project_mention_id": project_mention_id,
                            "event_id": event_id,
                            "identificador_boe": extraction.boe_id,
                            "fecha_publicacion": extraction.publication_date,
                            "alias_index": alias_idx,
                            "alias": alias,
                            "alias_norm": alias_norm,
                        }
                    )

    if not records:
        return empty_df(PROJECT_ALIASES_COLUMNS)

    return pd.DataFrame(
        records,
        columns=PROJECT_ALIASES_COLUMNS,
    )

#### flatten_associated_infrastructure

In [57]:
def flatten_associated_infrastructure(
    ai_extractions: pd.DataFrame,
) -> pd.DataFrame:
    """
    Aplana el resumen de infraestructura asociada de cada evento publicado.

    Esta tabla no descompone líneas, subestaciones o posiciones eléctricas. Solo
    conserva la síntesis auxiliar extraída en `AssociatedInfrastructureSummary`.
    """
    records = []

    for extraction in iter_valid_extractions(ai_extractions):
        for event_idx, event in enumerate(
            extraction.publication_events,
            start=1,
        ):
            infra = event.associated_infrastructure

            if infra is None:
                continue

            has_content = any(
                [
                    infra.has_evacuation_infrastructure is not None,
                    infra.has_electrical_substation is not None,
                    infra.has_grid_connection is not None,
                    infra.has_shared_infrastructure is not None,
                    safe_str(infra.description) != "",
                    safe_str(infra.evidence) != "",
                ]
            )

            if not has_content:
                continue

            event_id = make_event_id(
                extraction.boe_id,
                event_idx,
            )

            records.append(
                {
                    "associated_infrastructure_id": (
                        f"{event_id}_associated_infrastructure"
                    ),
                    "event_id": event_id,
                    "identificador_boe": extraction.boe_id,
                    "fecha_publicacion": extraction.publication_date,
                    "has_evacuation_infrastructure": (
                        infra.has_evacuation_infrastructure
                    ),
                    "has_electrical_substation": (
                        infra.has_electrical_substation
                    ),
                    "has_grid_connection": infra.has_grid_connection,
                    "has_shared_infrastructure": infra.has_shared_infrastructure,
                    "description": infra.description,
                    "evidence": infra.evidence,
                }
            )

    if not records:
        return empty_df(ASSOCIATED_INFRASTRUCTURE_COLUMNS)

    return pd.DataFrame(
        records,
        columns=ASSOCIATED_INFRASTRUCTURE_COLUMNS,
    )

### Generar tablas derivadas

In [58]:
ai_extractions = load_ai_extractions(
    BOE_AI_EXTRACTIONS_PATH
)

publication_events = flatten_publication_events(ai_extractions)
administrative_actions = flatten_administrative_actions(ai_extractions)
project_mentions = flatten_project_mentions(ai_extractions)
project_technical_attributes = flatten_project_technical_attributes(ai_extractions)
project_participants = flatten_project_participants(ai_extractions)
project_locations = flatten_project_locations(ai_extractions)
project_aliases = flatten_project_aliases(ai_extractions)
associated_infrastructure = flatten_associated_infrastructure(ai_extractions)

### Guardar tablas derivadas

In [59]:
save_parquet(publication_events, PUBLICATION_EVENTS_PATH)
save_parquet(administrative_actions, ADMINISTRATIVE_ACTIONS_PATH)
save_parquet(project_mentions, PROJECT_MENTIONS_PATH)
save_parquet(project_technical_attributes, PROJECT_TECHNICAL_ATTRIBUTES_PATH)
save_parquet(project_participants, PROJECT_PARTICIPANTS_PATH)
save_parquet(project_locations, PROJECT_LOCATIONS_PATH)
save_parquet(project_aliases, PROJECT_ALIASES_PATH)
save_parquet(associated_infrastructure, ASSOCIATED_INFRASTRUCTURE_PATH)

### Recargar tablas derivadas

In [60]:
publication_events = pd.read_parquet(PUBLICATION_EVENTS_PATH)
administrative_actions = pd.read_parquet(ADMINISTRATIVE_ACTIONS_PATH)
project_mentions = pd.read_parquet(PROJECT_MENTIONS_PATH)
project_technical_attributes = pd.read_parquet(PROJECT_TECHNICAL_ATTRIBUTES_PATH)
project_participants = pd.read_parquet(PROJECT_PARTICIPANTS_PATH)
project_locations = pd.read_parquet(PROJECT_LOCATIONS_PATH)
project_aliases = pd.read_parquet(PROJECT_ALIASES_PATH)
associated_infrastructure = pd.read_parquet(ASSOCIATED_INFRASTRUCTURE_PATH)

### Inspección rápida

In [61]:
display(publication_events)
display(administrative_actions)
display(project_mentions)
display(project_technical_attributes)
display(project_participants)
display(project_locations)
display(project_aliases)
display(associated_infrastructure)

,event_id,identificador_boe,fecha_publicacion,event_index,event_type,event_summary,evidence
0,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,proyecto_nuevo,Se somete a información pública la solicitud d...,Anuncio del Área de Industria y Energía de la ...
1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,hibridacion,Formulación de la declaración de impacto ambie...,"Resolución de 22 de diciembre de 2022, de la D..."
2,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,proyecto_nuevo,Formulación de la declaración de impacto ambie...,"Resolución de 23 de enero de 2023, de la Direc..."
3,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,proyecto_nuevo,Resolución por la que se otorga autorización a...,"Resolución de 17 de abril de 2023, de la Direc..."
4,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,modificacion,Información pública de la modificación de la A...,Anuncio del Área Funcional de Industria y Ener...
5,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,proyecto_nuevo,Formulación del informe de determinación de af...,"Resolución de 6 de mayo de 2024, de la Direcci..."
6,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,hibridacion,Sometimiento a información pública de la solic...,Anuncio del Área de Industria y Energía de la ...
7,BOE-A-2024-16664_event_1,BOE-A-2024-16664,2024-08-10,1,modificacion,Otorgamiento de autorización administrativa pr...,"Resolución de 22 de julio de 2024, de la Direc..."
8,BOE-A-2025-18285_event_1,BOE-A-2025-18285,2025-09-15,1,incorporacion_almacenamiento,Autorización administrativa previa y de constr...,"Resolución de 7 de agosto de 2025, de la Direc..."
9,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,1,proyecto_nuevo,Otorgamiento de autorización administrativa pr...,"Resolución de 17 de noviembre de 2025, de la D..."


,action_id,event_id,identificador_boe,fecha_publicacion,action_index,procedure_stage,procedure_decision,evidence
0,BOE-B-2021-32560_event_1_action_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,autorizacion_administrativa_previa,solicitado,solicitud de Autorización Administrativa Previa
1,BOE-B-2021-32560_event_1_action_2,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,2,informacion_publica,sometido_informacion_publica,"se somete al trámite de información pública, d..."
2,BOE-A-2022-24404_event_1_action_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,declaracion_impacto_ambiental,formulado,"Resolución de 22 de diciembre de 2022, de la D..."
3,BOE-A-2023-2598_event_1_action_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,declaracion_impacto_ambiental,formulado,"Resolución de 23 de enero de 2023, de la Direc..."
4,BOE-A-2023-10306_event_1_action_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,autorizacion_administrativa_previa,autorizado,"se otorga a Enel Green Power España, SL, autor..."
5,BOE-B-2023-19082_event_1_action_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,informacion_publica,sometido_informacion_publica,"se somete al trámite de información pública, d..."
6,BOE-B-2023-19082_event_1_action_2,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,2,modificacion,solicitado,la modificación de la Autorización Administrat...
7,BOE-B-2023-19082_event_1_action_3,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,3,autorizacion_administrativa_construccion,solicitado,la solicitud de la Autorización Administrativa...
8,BOE-A-2024-9608_event_1_action_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,informe_determinacion_afeccion_ambiental,formulado,por la que se formula informe de determinación...
9,BOE-B-2024-26379_event_1_action_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,solicitud_tramitacion,solicitado,solicitud de autorización administrativa previ...


,project_mention_id,event_id,identificador_boe,fecha_publicacion,local_project_id,project_name,project_name_norm,role_in_event,status_in_document,case_file_reference,evidence
0,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,project_1,Parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,PEol-416,Parque Eólico Badulaque de 90 MW y su infraest...
1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,project_1,Planta Fotovoltaica Híbrida «Majal Alto»,planta fotovoltaica hibrida majal alto,objeto_principal,en_tramitacion,None,"proyecto ""Planta fotovoltaica híbrida Majal Al..."
2,BOE-A-2022-24404_event_1_project_2,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,project_2,Parque Eólico Majal Alto,parque eolico majal alto,referencia_existente,en_explotacion,None,Planta Fotovoltaica «Majal Alto» para consegui...
3,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,project_1,None,,desconocido,desconocido,None,"Parque eólico Badulaque de 90 MW, y su infraes..."
4,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,project_1,Parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,None,El proyecto de Badulaque
5,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,project_1,parque eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,None,parque eólico Badulaque de 90 MW
6,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,project_1,Parque Eólico Badulaque,parque eolico badulaque,objeto_principal,en_tramitacion,PEol-416,PROYECTO MODIFICADO mayo 2023 de la instalació...
7,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,project_1,Instalación solar FV La Puebla 1,instalacion solar fv la puebla 1,objeto_principal,en_tramitacion,None,"Instalación solar FV La Puebla 1, de 100 MW de..."
8,BOE-A-2024-9608_event_1_project_2,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,project_2,PSFV La Puebla 2,psfv la puebla 2,referencia_asociada,en_tramitacion,None,"PSFV La Puebla 2, instalación del mismo promot..."
9,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,project_1,BESS Hibridación FV Andévalo,bess hibridacion fv andevalo,objeto_principal,en_tramitacion,PFOT-ALM-045,"instalación de almacenamiento por baterías ""BE..."


,project_technical_attribute_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,attribute_index,installation_type,installed_power_mw,peak_power_mwp,storage_power_mw,storage_capacity_mwh,description,power_normalization_note,evidence
0,BOE-B-2021-32560_event_1_project_1_attribute_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,eolica,90.00,NaN,None,None,Parque eólico integrado por veinte (20) aeroge...,None,Parque eólico Badulaque de 90 MW de potencia n...
1,BOE-A-2022-24404_event_1_project_1_attribute_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,fotovoltaica,43.50,50.447,None,None,"Estará compuesta por 8 recintos, con un períme...","Potencia nominal de 43,5 MWn y potencia pico d...",Planta fotovoltaica. Estará compuesta por 8 re...
2,BOE-A-2022-24404_event_1_project_2_attribute_1,BOE-A-2022-24404_event_1_project_2,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,eolica,NaN,NaN,None,None,None,None,Parque Eólico Majal Alto
3,BOE-A-2023-2598_event_1_project_1_attribute_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,eolica,90.00,NaN,None,None,20 aerogeneradores titulares y 8 aerogenerador...,None,El PE Badulaque está integrado por 20 aerogene...
4,BOE-A-2023-10306_event_1_project_1_attribute_1,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,eolica,90.00,NaN,None,None,None,None,parque eólico Badulaque de 90 MW
5,BOE-B-2023-19082_event_1_project_1_attribute_1,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,eolica,102.40,NaN,None,None,None,None,"Parque Eólico Badulaque», de 102,4 MW de poten..."
6,BOE-A-2024-9608_event_1_project_1_attribute_1,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,fotovoltaica,100.00,NaN,None,None,planta solar fotovoltaica,Potencia de 100 MW mencionada explícitamente c...,Instalación solar FV La Puebla 1 de 100 MW de ...
7,BOE-A-2024-9608_event_1_project_2_attribute_1,BOE-A-2024-9608_event_1_project_2,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,fotovoltaica,NaN,NaN,None,None,instalación fotovoltaica,None,PSFV La Puebla 2
8,BOE-B-2024-26379_event_1_project_1_attribute_1,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,almacenamiento,26.36,NaN,None,None,Sistema de almacenamiento de energía eléctrica...,None,"instalación de almacenamiento por baterías ""BE..."
9,BOE-B-2024-26379_event_1_project_2_attribute_1,BOE-B-2024-26379_event_1_project_2,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,fotovoltaica,42.56,NaN,None,None,None,None,"parque solar fotovoltaico existente, ""FV Andév..."


,project_participant_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,participant_index,participant_name,participant_name_norm,participant_role,evidence
0,BOE-B-2021-32560_event_1_project_1_participant_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,"ENEL GREEN POWER ESPAÑA, S.L.",enel green power espana s l,promotor,"Peticionario: ENEL GREEN POWER ESPAÑA, S.L."
1,BOE-A-2022-24404_event_1_project_1_participant_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,Iberdrola Renovables Andalucía,iberdrola renovables andalucia,promotor,promovido por Iberdrola Renovables Andalucía
2,BOE-A-2022-24404_event_1_project_1_participant_2,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,2,Dirección General de Política Energética y Min...,direccion general de politica energetica y min...,organo_sustantivo,la Dirección General de Política Energética y ...
3,BOE-A-2022-24404_event_1_project_1_participant_3,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,3,Dirección General de Calidad y Evaluación Ambi...,direccion general de calidad y evaluacion ambi...,organo_ambiental,"Resolución de 22 de diciembre de 2022, de la D..."
4,BOE-A-2023-2598_event_1_project_1_participant_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,Enel Green Power S.L.,enel green power s l,promotor,promovido por Enel Green Power S.L.
5,BOE-A-2023-2598_event_1_project_1_participant_2,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,2,Dirección General de Política Energética y Min...,direccion general de politica energetica y min...,organo_sustantivo,respecto de la que la Dirección General de Pol...
6,BOE-A-2023-2598_event_1_project_1_participant_3,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,3,Dirección General de Calidad y Evaluación Ambi...,direccion general de calidad y evaluacion ambi...,organo_ambiental,"Resolución de 23 de enero de 2023, de la Direc..."
7,BOE-A-2023-10306_event_1_project_1_participant_1,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,1,"Enel Green Power España, SL",enel green power espana sl,promotor,"se otorga a Enel Green Power España, SL, autor..."
8,BOE-B-2023-19082_event_1_project_1_participant_1,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,"ENEL GREEN POWER ESPAÑA, S.L. (unipersonal)",enel green power espana s l unipersonal,promotor,promovido por la mercantil «ENEL GREEN POWER E...
9,BOE-A-2024-9608_event_1_project_1_participant_1,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,"Jinko Greenfield Spain 3, SL",jinko greenfield spain 3 sl,promotor,"promovido por Jinko Greenfield Spain 3, SL"


,project_location_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,location_index,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
0,BOE-B-2021-32560_event_1_project_1_location_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,As Pontes,as pontes,A Coruña,a coruna,None,,Municipios afectados: As Pontes
1,BOE-B-2021-32560_event_1_project_1_location_2,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,2,As Somozas,as somozas,A Coruña,a coruna,None,,Municipios afectados: As Somozas
2,BOE-B-2021-32560_event_1_project_1_location_3,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,3,Cedeira,cedeira,A Coruña,a coruna,None,,Municipios afectados: Cedeira
3,BOE-B-2021-32560_event_1_project_1_location_4,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,4,Cerdido,cerdido,A Coruña,a coruna,None,,Municipios afectados: Cerdido
4,BOE-B-2021-32560_event_1_project_1_location_5,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,5,Moeche,moeche,A Coruña,a coruna,None,,Municipios afectados: Moeche
5,BOE-B-2021-32560_event_1_project_1_location_6,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,6,Valdovino,valdovino,A Coruña,a coruna,None,,Municipios afectados: Valdovino
6,BOE-B-2021-32560_event_1_project_1_location_7,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,7,A Coruña,a coruna,A Coruña,a coruna,Galicia,galicia,provincia de A Coruña
7,BOE-A-2022-24404_event_1_project_1_location_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,Puebla de Guzmán,puebla de guzman,Huelva,huelva,None,,en el municipio de Puebla de Guzmán en Huelva
8,BOE-A-2022-24404_event_1_project_2_location_1,BOE-A-2022-24404_event_1_project_2,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,Puebla de Guzmán,puebla de guzman,Huelva,huelva,None,,en el municipio de Puebla de Guzmán en Huelva
9,BOE-A-2023-2598_event_1_project_1_location_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,Valdoviño,valdovino,A Coruña,a coruna,None,,"en los Concellos de Valdoviño, Cerdido, Cerdei..."


,project_alias_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,alias_index,alias,alias_norm
0,BOE-B-2021-32560_event_1_project_1_alias_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,Parque eólico Badulaque de 90 MW,parque eolico badulaque de 90 mw
1,BOE-A-2022-24404_event_1_project_1_alias_1,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,1,Planta Fotovoltaica «Majal Alto»,planta fotovoltaica majal alto
2,BOE-A-2022-24404_event_1_project_1_alias_2,BOE-A-2022-24404_event_1_project_1,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,2,Planta Solar Fotovoltaica «FV Majal Alto»,planta solar fotovoltaica fv majal alto
3,BOE-A-2023-2598_event_1_project_1_alias_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,PE Badulaque,pe badulaque
4,BOE-A-2023-2598_event_1_project_1_alias_1,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,1,PE Badulaque,pe badulaque
5,BOE-B-2023-19082_event_1_project_1_alias_1,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,1,PROYECTO MODIFICADO mayo 2023 de la instalació...,proyecto modificado mayo 2023 de la instalacio...
6,BOE-B-2023-19082_event_1_project_1_alias_2,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,2,código GRE.EEC.R.73.ES.W.14487.00.001.00,codigo gre eec r 73 es w 14487 00 001 00
7,BOE-B-2023-19082_event_1_project_1_alias_3,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,3,núm. visado 230854,num visado 230854
8,BOE-A-2024-9608_event_1_project_1_alias_1,BOE-A-2024-9608_event_1_project_1,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,1,PSFV La Puebla 1,psfv la puebla 1
9,BOE-B-2024-26379_event_1_project_1_alias_1,BOE-B-2024-26379_event_1_project_1,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,1,Sistema de almacenamiento de energía eléctrica...,sistema de almacenamiento de energia electrica...


,associated_infrastructure_id,event_id,identificador_boe,fecha_publicacion,has_evacuation_infrastructure,has_electrical_substation,has_grid_connection,has_shared_infrastructure,description,evidence
0,BOE-B-2021-32560_event_1_associated_infrastruc...,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,True,True,True,True,La infraestructura de evacuación incluye 5 lín...,infraestructura de evacuación; La evacuación d...
1,BOE-A-2022-24404_event_1_associated_infrastruc...,BOE-A-2022-24404_event_1,BOE-A-2022-24404,2022-12-30,True,True,False,True,Línea eléctrica de evacuación soterrada de 20 ...,Línea eléctrica de evacuación. La energía se e...
2,BOE-A-2023-2598_event_1_associated_infrastructure,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,None,None,None,None,"Infraestructura de evacuación asociada, red de...",y su infraestructura de evacuación asociada
3,BOE-A-2023-10306_event_1_associated_infrastruc...,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,True,None,None,None,infraestructuras de evacuación del parque eóli...,y sus infraestructuras de evacuación
4,BOE-B-2023-19082_event_1_associated_infrastruc...,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,True,False,False,False,accesos y su infraestructura de evacuación,sus accesos y su infraestructura de evacuación
5,BOE-A-2024-9608_event_1_associated_infrastructure,BOE-A-2024-9608_event_1,BOE-A-2024-9608,2024-05-13,True,True,True,True,Infraestructura de evacuación compuesta por un...,La energía producida evacuará a través de la n...
6,BOE-B-2024-26379_event_1_associated_infrastruc...,BOE-B-2024-26379_event_1,BOE-B-2024-26379,2024-07-13,True,True,True,True,Infraestructura de evacuación para la hibridac...,"y su infraestructura de evacuación"", ""Esos dos..."
7,BOE-A-2024-16664_event_1_associated_infrastruc...,BOE-A-2024-16664_event_1,BOE-A-2024-16664,2024-08-10,True,False,False,False,"Infraestructuras de evacuación, incluyendo una...","y sus infraestructuras de evacuación, ubicados..."
8,BOE-A-2025-18285_event_1_associated_infrastruc...,BOE-A-2025-18285_event_1,BOE-A-2025-18285,2025-09-15,None,None,None,None,"infraestructura de evacuación, consistente en ...","y para su infraestructura de evacuación, consi..."
9,BOE-A-2025-26110_event_1_associated_infrastruc...,BOE-A-2025-26110_event_1,BOE-A-2025-26110,2025-12-19,True,True,True,False,Infraestructura de evacuación consistente en l...,"y su infraestructura de evacuación, en Puebla ..."


In [62]:
# Chequeos básicos

print(f"{len(publication_events)=}")
print(f"{len(administrative_actions)=}")
print(f"{len(project_mentions)=}")
print(f"{len(project_technical_attributes)=}")
print(f"{len(project_participants)=}")
print(f"{len(project_locations)=}")
print(f"{len(project_aliases)=}")
print(f"{len(associated_infrastructure)=}")

len(publication_events)=13
len(administrative_actions)=26
len(project_mentions)=21
len(project_technical_attributes)=20
len(project_participants)=28
len(project_locations)=59
len(project_aliases)=16
len(associated_infrastructure)=13


In [63]:
# Ejemplo de revisión de localizaciones

project_locations.loc[
    project_locations["municipality_raw_norm"].str.contains(
        "pontes",
        na=False,
    )
]

,project_location_id,project_mention_id,event_id,identificador_boe,fecha_publicacion,location_index,municipality_raw,municipality_raw_norm,province_hint_raw,province_hint_raw_norm,autonomous_community_hint_raw,autonomous_community_hint_raw_norm,location_evidence
0,BOE-B-2021-32560_event_1_project_1_location_1,BOE-B-2021-32560_event_1_project_1,BOE-B-2021-32560_event_1,BOE-B-2021-32560,2021-07-07,1,As Pontes,as pontes,A Coruña,a coruna,None,,Municipios afectados: As Pontes
14,BOE-A-2023-2598_event_1_project_1_location_6,BOE-A-2023-2598_event_1_project_1,BOE-A-2023-2598_event_1,BOE-A-2023-2598,2023-01-31,6,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,None,,"en los Concellos de Valdoviño, Cerdido, Cerdei..."
20,BOE-A-2023-10306_event_1_project_1_location_6,BOE-A-2023-10306_event_1_project_1,BOE-A-2023-10306_event_1,BOE-A-2023-10306,2023-04-28,6,As Pontés,as pontes,A Coruña,a coruna,None,,As Pontés (A Coruña)
26,BOE-B-2023-19082_event_1_project_1_location_6,BOE-B-2023-19082_event_1_project_1,BOE-B-2023-19082_event_1,BOE-B-2023-19082,2023-06-22,6,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,Galicia,galicia,As Pontes de García Rodríguez
35,BOE-A-2024-16664_event_1_project_1_location_6,BOE-A-2024-16664_event_1_project_1,BOE-A-2024-16664_event_1,BOE-A-2024-16664,2024-08-10,6,As Pontes de García Rodríguez,as pontes de garcia rodriguez,A Coruña,a coruna,Galicia,galicia,"y As Pontes de García Rodríguez, en la provinc..."
